In [8]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

def logistic(t, K, b):
    return K / (1 + np.exp(-b * (t - 2022)))

data_dict = {
    "Beijing":      [15000, 16500, 18000, 20000],
    "Shanghai":     [12300, 14500, 18000, 25000],
    "Guangzhou":    [8000, 9000, 10000, 12000],
    "Shenzhen":     [10000, 11500, 13000, 15000],
    "Chengdu":      [7000, 10000, 14000, 19000],
    "Chongqing":    [5800, 8500, 12000, 17000],
    "Zhangjiakou":  [2000, 7600, 22800, 50000],
    "Wuhu":         [500, 2000, 6000, 15000],
    "Shaoguan":     [300, 1500, 4000, 10000],
    "Hohhot":       [800, 6000, 15000, 125000],
    "Guiyang":      [500, 2000, 25000, 100000],
    "Qingyang":     [100, 5000, 50000, 120000],
    "Zhongwei":     [44000, 88000, 126000, 130000]
}

# 分组
group_rule = {
    "Beijing":"L0","Shanghai":"L0","Guangzhou":"L0","Shenzhen":"L0",
    "Chengdu":"L1","Chongqing":"L1","Zhangjiakou":"L1","Wuhu":"L1","Shaoguan":"L1",
    "Hohhot":"L2","Guiyang":"L2","Qingyang":"L2","Zhongwei":"L2"
}
# 倍率
k_ratio = {"L0":1.8,"L1":3.0,"L2":4.0}

t_hist = np.array([2022,2023,2024,2025])
t_pred = np.arange(2026,2036)
res_all = {}
param_info = {}

for city, s_hist in data_dict.items():
    s_hist = np.array(s_hist)
    s25 = s_hist[-1]
    g = group_rule[city]
    k_max = s25 * k_ratio[g]
    # 边界：K∈[s25, k_max]，b∈[0.005,0.5]
    bounds = ((s25, 0.005), (k_max, 0.5))
    p0 = [s25*1.5, 0.08]
    popt,_ = curve_fit(logistic, t_hist, s_hist, p0=p0, bounds=bounds, maxfev=20000)
    K,b = popt
    param_info[city] = {"饱和上限K(P)":round(K,2),"增速系数b":round(b,4)}
    s_pred = logistic(t_pred, K, b)
    res_all[city] = np.round(s_pred,2)

df_param = pd.DataFrame(param_info).T
df_pred = pd.DataFrame(res_all, index=t_pred)

print("====拟合参数（K远期饱和上限、b增长系数）====")
print(df_param)
print("\n====2026~2035逐年算力(P)====")
print(df_pred)

sum_year = df_pred.sum(axis=1)
print("\n====每年全部城市总算力汇总====")
print(sum_year.round(2))
print(f"\n2035年13城总算力：{sum_year[2035]:.2f} P")

====拟合参数（K远期饱和上限、b增长系数）====
              饱和上限K(P)   增速系数b
Beijing       29768.65  0.2289
Shanghai      26502.89  0.5000
Guangzhou     15635.29  0.3501
Shenzhen      19755.31  0.3586
Chengdu       19286.39  0.5000
Chongqing     17000.00  0.5000
Zhangjiakou   50000.00  0.2855
Wuhu          15000.00  0.2496
Shaoguan      10000.00  0.2543
Hohhot       125000.00  0.0790
Guiyang      100000.00  0.1399
Qingyang     120000.00  0.2328
Zhongwei     149524.51  0.5000

====2026~2035逐年算力(P)====
       Beijing  Shanghai  Guangzhou  Shenzhen   Chengdu  Chongqing  \
2026  21260.33  23343.67   12543.04  15953.98  16987.40   14973.55   
2027  22581.02  24492.43   13321.19  16936.00  17823.36   15710.41   
2028  23754.69  25245.97   13930.14  17697.05  18371.72   16193.76   
2029  24779.07  25726.03   14393.78  18270.67  18721.06   16501.69   
2030  25659.14  26026.21   14739.46  18694.02  18939.50   16694.23   
2031  26405.06  26211.71   14993.19  19001.63  19074.49   16813.22   
2032  27030.03  26325.

In [16]:
import numpy as np
import pandas as pd
import os

# ===================== 1. 基础配置 =====================
# 保存路径
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
os.makedirs(save_path, exist_ok=True)

# 结果列表强制清空，杜绝重复数据
result_data = []

# 城市层级划分
city_class = {
    "Beijing": "L0",
    "Shanghai": "L0",
    "Guangzhou": "L0",
    "Shenzhen": "L0",
    "Chengdu": "L1",
    "Chongqing": "L1",
    "Zhangjiakou": "L1",
    "Wuhu": "L1",
    "Shaoguan": "L1",
    "Hohhot (Helingeer)": "L2",
    "Guiyang (Gui'an New Area)": "L2",
    "Qingyang": "L2",
    "Zhongwei": "L2"
}
city_list = list(city_class.keys())

# 2025年基准初始值 [通用, 智能, 推理, 训练]（模型计算基准，不输出）
init_2025 = {
    "Beijing":                  [69.90, 30.10, 55.00, 45.00],
    "Shanghai":                 [50.00, 50.00, 50.00, 50.00],
    "Guangzhou":                [50.00, 50.00, 55.00, 45.00],
    "Shenzhen":                 [50.00, 50.00, 50.00, 50.00],
    "Chengdu":                  [40.00, 60.00, 15.00, 85.00],
    "Chongqing":                [50.00, 50.00, 15.00, 85.00],
    "Zhangjiakou":              [30.00, 70.00, 20.00, 80.00],
    "Wuhu":                     [30.00, 70.00, 55.00, 45.00],
    "Shaoguan":                 [30.00, 70.00, 55.00, 45.00],
    "Hohhot (Helingeer)":       [5.00, 95.00, 5.00, 95.00],
    "Guiyang (Gui'an New Area)":[2.00, 98.00, 7.00, 93.00],
    "Qingyang":                 [2.00, 98.00, 5.00, 95.00],
    "Zhongwei":                 [20.00, 80.00, 5.00, 95.00]
}

# 2030年目标占比
target_2030 = {
    "L0": [20.00, 80.00, 70.00, 30.00],
    "L1": [10.00, 90.00, 90.00, 10.00],
    "L2": [1.00, 99.00, 0.00, 0.00]
}

# ===================== 2. 时间范围：仅 2026 - 2035 =====================
year_range = list(range(2026, 2036))
# 时间映射：t=0→2025(基准)，t=1→2026，t=5→2030，t=10→2035
t_map = {year: (year - 2025) for year in year_range}

# ===================== 3. 端点锁定Logistic模型 =====================
def logistic_fixed_endpoints(t, start_val, end_val, k=1.5):
    """
    t=0  对应2025基准值
    t=5  对应2030目标值
    t≥5  2030及以后保持目标值不变
    """
    if t >= 5:
        return round(end_val, 2)
    x = t / 5
    sigmoid = 1 / (1 + np.exp(-k * (2 * x - 1)))
    res = start_val + (end_val - start_val) * sigmoid
    return round(res, 2)

# ===================== 4. 批量计算 =====================
for city in city_list:
    level = city_class[city]
    g25, a25, i25, tr25 = init_2025[city]
    g30, a30, i30, tr30 = target_2030[level]

    for year in year_range:
        t = t_map[year]
        # 通用算力
        g_pred = logistic_fixed_endpoints(t, g25, g30)
        # 智能算力（互补，总和100%）
        a_pred = round(100.0 - g_pred, 2)

        # 推理、训练算力
        if level in ("L0", "L1"):
            i_pred = logistic_fixed_endpoints(t, i25, i30)
            tr_pred = logistic_fixed_endpoints(t, tr25, tr30)
        else:
            # L2节点：推理/训练保持2025原值不变
            i_pred = i25
            tr_pred = tr25

        result_data.append({
            "城市": city,
            "节点层级": level,
            "年份": year,
            "通用算力占比(%)": g_pred,
            "智能算力占比(%)": a_pred,
            "推理算力占比(智算内,%)": i_pred,
            "训练算力占比(智算内,%)": tr_pred
        })

# ===================== 5. 生成数据表 & 校验 =====================
df_result = pd.DataFrame(result_data)
# 标准行数：13个城市 × 10个年份(2026-2035) = 130行
print(f"数据总行数：{len(df_result)} （标准应为130行）")

# ===================== 6. 保存文件 =====================
csv_name = "各城市2026-2035算力结构占比_Logistic预测.csv"
excel_name = "各城市2026-2035算力结构占比_Logistic预测.xlsx"

csv_path = os.path.join(save_path, csv_name)
excel_path = os.path.join(save_path, excel_name)

df_result.to_csv(csv_path, index=False, encoding="utf-8-sig")
df_result.to_excel(excel_path, index=False, engine="openpyxl")

print(f"\n✅ 数据已保存至指定目录")
print(f"CSV文件：{csv_path}")
print(f"Excel文件：{excel_path}")

# ===================== 7. 抽样校验（北京 2026-2035） =====================
print("\n" + "="*70)
print("【抽样校验】北京 L0 节点 2026-2035 算力占比")
print("="*70)
df_beijing = df_result[df_result["城市"] == "Beijing"]
print(df_beijing.to_string(index=False))

数据总行数：130 （标准应为130行）

✅ 数据已保存至指定目录
CSV文件：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\各城市2026-2035算力结构占比_Logistic预测.csv
Excel文件：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\各城市2026-2035算力结构占比_Logistic预测.xlsx

【抽样校验】北京 L0 节点 2026-2035 算力占比
     城市 节点层级   年份  通用算力占比(%)  智能算力占比(%)  推理算力占比(智算内,%)  训练算力占比(智算内,%)
Beijing   L0 2026      55.48      44.52          59.34          40.66
Beijing   L0 2027      48.66      51.34          61.38          38.62
Beijing   L0 2028      41.24      58.76          63.62          36.38
Beijing   L0 2029      34.42      65.58          65.66          34.34
Beijing   L0 2030      20.00      80.00          70.00          30.00
Beijing   L0 2031      20.00      80.00          70.00          30.00
Beijing   L0 2032      20.00      80.00          70.00          30.00
Beijing   L0 2033      20.00      80.00          70.00          30.00
Beijing   L0 2034      20.00      80.00          70.00          30.00
Beijing   L0 2035      20.00      80.00          70.00          30.00


In [17]:
import pandas as pd
import numpy as np
import os

# ===================== 1. 基础配置 =====================
# 数据保存路径（和你之前的路径完全一致）
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
os.makedirs(save_path, exist_ok=True)

# 城市层级划分（和之前完全一致，保证匹配）
city_class = {
    "Beijing": "L0",
    "Shanghai": "L0",
    "Guangzhou": "L0",
    "Shenzhen": "L0",
    "Chengdu": "L1",
    "Chongqing": "L1",
    "Zhangjiakou": "L1",
    "Wuhu": "L1",
    "Shaoguan": "L1",
    "Hohhot (Helingeer)": "L2",
    "Guiyang (Gui'an New Area)": "L2",
    "Qingyang": "L2",
    "Zhongwei": "L2"
}
city_list = list(city_class.keys())

# 2025年基准初始占比 [通用, 智能, 推理, 训练]
init_2025 = {
    "Beijing":                  [69.90, 30.10, 55.00, 45.00],
    "Shanghai":                 [50.00, 50.00, 50.00, 50.00],
    "Guangzhou":                [50.00, 50.00, 55.00, 45.00],
    "Shenzhen":                 [50.00, 50.00, 50.00, 50.00],
    "Chengdu":                  [40.00, 60.00, 15.00, 85.00],
    "Chongqing":                [50.00, 50.00, 15.00, 85.00],
    "Zhangjiakou":              [30.00, 70.00, 20.00, 80.00],
    "Wuhu":                     [30.00, 70.00, 55.00, 45.00],
    "Shaoguan":                 [30.00, 70.00, 55.00, 45.00],
    "Hohhot (Helingeer)":       [5.00, 95.00, 5.00, 95.00],
    "Guiyang (Gui'an New Area)":[2.00, 98.00, 7.00, 93.00],
    "Qingyang":                 [2.00, 98.00, 5.00, 95.00],
    "Zhongwei":                 [20.00, 80.00, 5.00, 95.00]
}

# 2030年目标占比
target_2030 = {
    "L0": [20.00, 80.00, 70.00, 30.00],
    "L1": [10.00, 90.00, 90.00, 10.00],
    "L2": [1.00, 99.00, 0.00, 0.00]
}

# 时间范围：2026-2035
year_range = list(range(2026, 2036))
t_map = {year: (year - 2025) for year in year_range}

# ===================== 2. 生成2026-2035算力结构占比表 =====================
def logistic_fixed_endpoints(t, start_val, end_val, k=1.5):
    if t >= 5:
        return round(end_val, 2)
    x = t / 5
    sigmoid = 1 / (1 + np.exp(-k * (2 * x - 1)))
    res = start_val + (end_val - start_val) * sigmoid
    return round(res, 2)

# 生成占比长表
ratio_data = []
for city in city_list:
    level = city_class[city]
    g25, a25, i25, tr25 = init_2025[city]
    g30, a30, i30, tr30 = target_2030[level]
    for year in year_range:
        t = t_map[year]
        g_pred = logistic_fixed_endpoints(t, g25, g30)
        a_pred = round(100.0 - g_pred, 2)
        if level in ("L0", "L1"):
            i_pred = logistic_fixed_endpoints(t, i25, i30)
            tr_pred = logistic_fixed_endpoints(t, tr25, tr30)
        else:
            i_pred = i25
            tr_pred = tr25
        ratio_data.append({
            "城市": city,
            "节点层级": level,
            "年份": year,
            "通用算力占比(%)": g_pred,
            "智能算力占比(%)": a_pred,
            "推理算力占比(智算内,%)": i_pred,
            "训练算力占比(智算内,%)": tr_pred
        })
df_ratio = pd.DataFrame(ratio_data)

# ===================== 3. 录入你提供的总算力规模数据 =====================
# 严格按你提供的数值录入，城市名称完全匹配
total_power_data = [
    {"City": "Beijing", "2026": 22000, "2027": 24200, "2028": 26620, "2029": 29282, "2030": 32210.2, "2031": 33498.608, "2032": 34838.55232, "2033": 36232.09441, "2034": 37681.37819, "2035": 39188.63332},
    {"City": "Shanghai", "2026": 27500, "2027": 30250, "2028": 33275, "2029": 36602.5, "2030": 40262.75, "2031": 41873.26, "2032": 43548.1904, "2033": 45290.11802, "2034": 47101.72274, "2035": 48985.79165},
    {"City": "Guangzhou", "2026": 13200, "2027": 14520, "2028": 15972, "2029": 17569.2, "2030": 19326.12, "2031": 20099.1648, "2032": 20903.13139, "2033": 21739.25665, "2034": 22608.82691, "2035": 23513.17999},
    {"City": "Shenzhen", "2026": 16500, "2027": 18150, "2028": 19965, "2029": 21961.5, "2030": 24157.65, "2031": 25123.956, "2032": 26128.91424, "2033": 27174.07081, "2034": 28261.03364, "2035": 29391.47499},
    {"City": "Chengdu", "2026": 27778, "2027": 40055.876, "2028": 56959.45567, "2029": 79857.15685, "2030": 110362.5908, "2031": 119191.598, "2032": 128726.9259, "2033": 139025.0799, "2034": 150147.0863, "2035": 162158.8532},
    {"City": "Chongqing", "2026": 24854, "2027": 35839.468, "2028": 50963.7235, "2029": 71451.14034, "2030": 98745.47595, "2031": 106645.114, "2032": 115176.7232, "2033": 124390.861, "2034": 134342.1299, "2035": 145089.5003},
    {"City": "Zhangjiakou", "2026": 73100, "2027": 105410.2, "2028": 149893.3044, "2029": 210150.4128, "2030": 290427.8704, "2031": 313662.1001, "2032": 338755.0681, "2033": 365855.4735, "2034": 395123.9114, "2035": 426733.8243},
    {"City": "Wuhu", "2026": 38012, "2027": 54813.304, "2028": 77944.51829, "2029": 109278.2146, "2030": 151022.4926, "2031": 163104.292, "2032": 176152.6354, "2033": 190244.8462, "2034": 205464.4339, "2035": 221901.5887},
    {"City": "Shaoguan", "2026": 14620, "2027": 21082.04, "2028": 29978.66088, "2029": 42030.08255, "2030": 58085.57409, "2031": 62732.42002, "2032": 67751.01362, "2033": 73171.09471, "2034": 79024.78228, "2035": 85346.76487},
    {"City": "Hohhot (Helingeer)", "2026": 195250, "2027": 304980.5, "2028": 476379.541, "2029": 744104.843, "2030": 1162291.765, "2031": 1301766.777, "2032": 1457978.79, "2033": 1632936.245, "2034": 1828888.594, "2035": 2048355.225},
    {"City": "Guiyang (Gui'an New Area)", "2026": 156200, "2027": 243984.4, "2028": 381103.6328, "2029": 595283.8744, "2030": 929833.4119, "2031": 1041413.421, "2032": 1166383.032, "2033": 1306348.996, "2034": 1463110.875, "2035": 1638684.18},
    {"City": "Qingyang", "2026": 187440, "2027": 292781.28, "2028": 457324.3594, "2029": 714340.6493, "2030": 1115800.094, "2031": 1249696.106, "2032": 1399659.638, "2033": 1567618.795, "2034": 1755733.05, "2035": 1966421.016},
    {"City": "Zhongwei", "2026": 203060, "2027": 317179.72, "2028": 495434.7226, "2029": 773869.0368, "2030": 1208783.435, "2031": 1353837.448, "2032": 1516297.941, "2033": 1698253.694, "2034": 1902044.138, "2035": 2130289.434}
]

# 总算力宽表转长表，方便合并
df_total_power = pd.DataFrame(total_power_data)
df_total_power = df_total_power.melt(id_vars=["City"], var_name="年份", value_name="总算力(PFLOPS)")
df_total_power["年份"] = df_total_power["年份"].astype(int)
# 列名统一，和占比表匹配
df_total_power = df_total_power.rename(columns={"City": "城市"})

# ===================== 4. 合并占比表 + 总算力表，计算各类型算力绝对规模 =====================
# 按城市、年份合并
df_merge = pd.merge(df_ratio, df_total_power, on=["城市", "年份"], how="left")

# 计算各类型算力绝对规模（单位：PFLOPS，保留2位小数）
df_merge["通用算力(PFLOPS)"] = round(df_merge["总算力(PFLOPS)"] * df_merge["通用算力占比(%)"] / 100, 2)
df_merge["智能算力(PFLOPS)"] = round(df_merge["总算力(PFLOPS)"] * df_merge["智能算力占比(%)"] / 100, 2)
df_merge["推理算力(PFLOPS)"] = round(df_merge["智能算力(PFLOPS)"] * df_merge["推理算力占比(智算内,%)"] / 100, 2)
df_merge["训练算力(PFLOPS)"] = round(df_merge["智能算力(PFLOPS)"] * df_merge["训练算力占比(智算内,%)"] / 100, 2)

# 第一个表格：基础算力绝对规模表
df_base = df_merge[[
    "城市", "节点层级", "年份", "总算力(PFLOPS)",
    "通用算力(PFLOPS)", "智能算力(PFLOPS)", "推理算力(PFLOPS)", "训练算力(PFLOPS)"
]].copy()

# ===================== 5. 生成第二个表格：添加指定衍生列 =====================
df_derived = df_base.copy()

# 定义衍生列计算规则：仅L0城市生效，非L0为0
# 列1：L0推理算力 × 30%
df_derived["L0推理算力×30%(PFLOPS)"] = np.where(
    df_derived["节点层级"] == "L0",
    round(df_derived["推理算力(PFLOPS)"] * 0.3, 2),
    0
)

# 列2：L0训练算力 × 100%
df_derived["L0训练算力×100%(PFLOPS)"] = np.where(
    df_derived["节点层级"] == "L0",
    round(df_derived["训练算力(PFLOPS)"] * 1.0, 2),
    0
)

# 列3：L0推理算力 × 50%
df_derived["L0推理算力×50%(PFLOPS)"] = np.where(
    df_derived["节点层级"] == "L0",
    round(df_derived["推理算力(PFLOPS)"] * 0.5, 2),
    0
)

# 列4：L0推理算力 × 10%
df_derived["L0推理算力×10%(PFLOPS)"] = np.where(
    df_derived["节点层级"] == "L0",
    round(df_derived["推理算力(PFLOPS)"] * 0.1, 2),
    0
)

# ===================== 6. 保存文件到指定路径 =====================
# 保存第一个表格：基础算力绝对规模
base_csv_path = os.path.join(save_path, "各城市2026-2035各类型算力绝对规模.csv")
base_excel_path = os.path.join(save_path, "各城市2026-2035各类型算力绝对规模.xlsx")
df_base.to_csv(base_csv_path, index=False, encoding="utf-8-sig")
df_base.to_excel(base_excel_path, index=False, engine="openpyxl")

# 保存第二个表格：含衍生列的完整表
derived_csv_path = os.path.join(save_path, "各城市2026-2035算力规模_含L0衍生计算列.csv")
derived_excel_path = os.path.join(save_path, "各城市2026-2035算力规模_含L0衍生计算列.xlsx")
df_derived.to_csv(derived_csv_path, index=False, encoding="utf-8-sig")
df_derived.to_excel(derived_excel_path, index=False, engine="openpyxl")

# ===================== 7. 输出结果预览 =====================
print("="*100)
print("✅ 所有表格生成完成，已保存到指定路径！")
print("="*100)
print("1. 基础算力绝对规模表：")
print(f"   CSV：{base_csv_path}")
print(f"   Excel：{base_excel_path}")
print("\n2. 含L0衍生计算列的完整表：")
print(f"   CSV：{derived_csv_path}")
print(f"   Excel：{derived_excel_path}")
print("="*100)
print("\n【预览：北京L0城市2026-2030基础算力规模】")
print(df_base[(df_base["城市"] == "Beijing") & (df_base["年份"] <= 2030)].to_string(index=False))
print("\n【预览：北京L0城市2026-2030衍生计算列】")
print(df_derived[(df_derived["城市"] == "Beijing") & (df_derived["年份"] <= 2030)][[
    "城市", "年份", "推理算力(PFLOPS)", "训练算力(PFLOPS)",
    "L0推理算力×30%(PFLOPS)", "L0训练算力×100%(PFLOPS)",
    "L0推理算力×50%(PFLOPS)", "L0推理算力×10%(PFLOPS)"
]].to_string(index=False))

✅ 所有表格生成完成，已保存到指定路径！
1. 基础算力绝对规模表：
   CSV：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\各城市2026-2035各类型算力绝对规模.csv
   Excel：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\各城市2026-2035各类型算力绝对规模.xlsx

2. 含L0衍生计算列的完整表：
   CSV：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\各城市2026-2035算力规模_含L0衍生计算列.csv
   Excel：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\各城市2026-2035算力规模_含L0衍生计算列.xlsx

【预览：北京L0城市2026-2030基础算力规模】
     城市 节点层级   年份  总算力(PFLOPS)  通用算力(PFLOPS)  智能算力(PFLOPS)  推理算力(PFLOPS)  训练算力(PFLOPS)
Beijing   L0 2026      22000.0      12205.60       9794.40       5812.00       3982.40
Beijing   L0 2027      24200.0      11775.72      12424.28       7626.02       4798.26
Beijing   L0 2028      26620.0      10978.09      15641.91       9951.38       5690.53
Beijing   L0 2029      29282.0      10078.86      19203.14      12608.78       6594.36
Beijing   L0 2030      32210.2       6442.04      25768.16      18037.71       7730.45

【预览：北京L0城市2026-2030衍生计算列】
     城市   年份  推理算力(PFLOPS)  训练算力(PFLOPS)  L0推理算力×30%(PFLOPS)  L0训练算力×100%(PFLOPS)  L0推理算

In [18]:
import pandas as pd
import numpy as np
import os

# ===================== 1. 基础配置 & 读取原始表格 =====================
# 数据保存路径（和你的路径完全一致）
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
os.makedirs(save_path, exist_ok=True)

# 读取你之前生成的含L0衍生列的原始表格
original_file = os.path.join(save_path, "各城市2026-2035算力规模_含L0衍生计算列.xlsx")
df = pd.read_excel(original_file)

# 城市名称标准化（确保匹配无错误）
city_name_fix = {
    "Hohhot": "Hohhot (Helingeer)",
    "Guiyang": "Guiyang (Gui'an New Area)"
}
df["城市"] = df["城市"].replace(city_name_fix)
year_list = df["年份"].unique()
city_list = df["城市"].unique()

# ===================== 2. 第一步：补充芜湖的衍生列计算（核心修正） =====================
# 定义需要补充的列和对应比例
wuhu_calc_rules = {
    "L0推理算力×30%(PFLOPS)": 0.3,
    "L0推理算力×50%(PFLOPS)": 0.5,
    "L0推理算力×10%(PFLOPS)": 0.1
}

# 给芜湖补充计算对应列
for col_name, ratio in wuhu_calc_rules.items():
    # 芜湖的对应列 = 芜湖的推理算力 × 比例
    df.loc[df["城市"] == "Wuhu", col_name] = round(
        df.loc[df["城市"] == "Wuhu", "推理算力(PFLOPS)"] * ratio, 2
    )

# ===================== 3. 第二步：定义通用转移计算函数 =====================
def calculate_transfer(df, source_col, transfer_rules):
    """
    通用转移计算函数：按规则计算转移规模，返回和df行数完全匹配的结果列
    :param df: 原始数据表
    :param source_col: 用于转移的基数列名
    :param transfer_rules: 转移规则字典，格式：{转出城市: {接收城市: 分配比例}}
    :return: 转移结果Series，保留2位小数
    """
    transfer_result = pd.Series(0.0, index=df.index)
    
    # 按年份分组，逐年计算（确保同年度转移匹配）
    for year in year_list:
        year_df = df[df["年份"] == year].copy()
        # 遍历每个转出城市的分配规则
        for from_city, to_rules in transfer_rules.items():
            if from_city not in year_df["城市"].values:
                continue
            # 获取该城市当年的转移基数
            source_value = year_df[year_df["城市"] == from_city][source_col].values[0]
            # 按比例分配给每个接收城市
            for to_city, ratio in to_rules.items():
                if to_city not in year_df["城市"].values:
                    continue
                transfer_value = source_value * ratio
                # 累加到接收城市的对应行
                to_index = year_df[year_df["城市"] == to_city].index[0]
                transfer_result.loc[to_index] += transfer_value
    
    return round(transfer_result, 2)

# ===================== 4. 第三步：定义更新后的转移规则 =====================
# 4.1 第一列：推理算力转移规模（基于30%推理列，补充芜湖规则）
rule_30_infer = {
    "Beijing": {"Zhangjiakou": 1.0},
    "Shanghai": {"Wuhu": 0.5, "Chengdu": 0.25, "Chongqing": 0.25},
    "Wuhu": {"Chengdu": 0.5, "Chongqing": 0.5},  # 新增芜湖规则
    "Guangzhou": {"Shaoguan": 1.0},
    "Shenzhen": {"Shaoguan": 1.0}
}

# 4.2 第二列：训练算力转移规模（基于100%训练列，规则不变）
rule_train = {
    "Beijing": {"Hohhot (Helingeer)": 0.6, "Zhongwei": 0.4},
    "Zhangjiakou": {"Hohhot (Helingeer)": 0.6, "Zhongwei": 0.4},
    "Shanghai": {"Qingyang": 1.0},
    "Wuhu": {"Qingyang": 1.0},
    "Chengdu": {"Qingyang": 0.5, "Guiyang (Gui'an New Area)": 0.5},
    "Chongqing": {"Qingyang": 0.5, "Guiyang (Gui'an New Area)": 0.5},
    "Guangzhou": {"Guiyang (Gui'an New Area)": 1.0},
    "Shenzhen": {"Guiyang (Gui'an New Area)": 1.0},
    "Shaoguan": {"Guiyang (Gui'an New Area)": 1.0}
}

# 4.3 第三列：50%推理转移规模（基于50%推理列，补充芜湖规则）
rule_50_infer = {
    "Beijing": {"Zhangjiakou": 1.0},
    "Shanghai": {"Wuhu": 0.5, "Chengdu": 0.25, "Chongqing": 0.25},
    "Wuhu": {"Chengdu": 0.5, "Chongqing": 0.5},  # 新增芜湖规则
    "Guangzhou": {"Shaoguan": 1.0},
    "Shenzhen": {"Shaoguan": 1.0}
}

# 4.4 第四列：10%推理转移规模（基于10%推理列，补充芜湖规则）
rule_10_infer = {
    "Beijing": {"Hohhot (Helingeer)": 0.6, "Zhongwei": 0.4},
    "Shanghai": {"Qingyang": 1.0},
    "Wuhu": {"Qingyang": 1.0},  # 新增芜湖规则
    "Guangzhou": {"Guiyang (Gui'an New Area)": 1.0},
    "Shenzhen": {"Guiyang (Gui'an New Area)": 1.0}
}

# ===================== 5. 第四步：计算4列新增转移规模并添加到表格 =====================
# 5.1 推理算力转移规模
df["推理算力转移规模(PFLOPS)"] = calculate_transfer(
    df, "L0推理算力×30%(PFLOPS)", rule_30_infer
)

# 5.2 训练算力转移规模
df["训练算力转移规模(PFLOPS)"] = calculate_transfer(
    df, "L0训练算力×100%(PFLOPS)", rule_train
)

# 5.3 50%推理转移规模
df["50%推理转移规模(PFLOPS)"] = calculate_transfer(
    df, "L0推理算力×50%(PFLOPS)", rule_50_infer
)

# 5.4 10%推理转移规模
df["10%推理转移规模(PFLOPS)"] = calculate_transfer(
    df, "L0推理算力×10%(PFLOPS)", rule_10_infer
)

# ===================== 6. 第五步：保存全新的完整表格 =====================
final_csv_path = os.path.join(save_path, "各城市2026-2035算力规模_含芜湖修正+转移计算完整表.csv")
final_excel_path = os.path.join(save_path, "各城市2026-2035算力规模_含芜湖修正+转移计算完整表.xlsx")

df.to_csv(final_csv_path, index=False, encoding="utf-8-sig")
df.to_excel(final_excel_path, index=False, engine="openpyxl")

# ===================== 7. 结果预览 & 校验 =====================
print("="*130)
print("✅ 芜湖衍生列补充完成 + 转移规模计算完成，全新完整表已保存！")
print("="*130)
print(f"最终完整表 CSV：{final_csv_path}")
print(f"最终完整表 Excel：{final_excel_path}")
print("="*130)

# 预览核心结果：2026年芜湖相关的转移结果
print("\n【核心结果预览：2026年芜湖相关转移汇总】")
preview_2026 = df[df["年份"] == 2026][[
    "城市", "L0推理算力×30%(PFLOPS)", "推理算力转移规模(PFLOPS)",
    "L0推理算力×10%(PFLOPS)", "10%推理转移规模(PFLOPS)"
]]
# 只显示有转移数据的城市
print(preview_2026[preview_2026.sum(axis=1, numeric_only=True) > 0].to_string(index=False))

# 预览芜湖2026年的衍生列计算结果
print("\n【芜湖衍生列修正结果预览：2026年】")
wuhu_2026 = df[(df["城市"] == "Wuhu") & (df["年份"] == 2026)][[
    "城市", "年份", "推理算力(PFLOPS)",
    "L0推理算力×30%(PFLOPS)", "L0推理算力×50%(PFLOPS)", "L0推理算力×10%(PFLOPS)"
]]
print(wuhu_2026.to_string(index=False))

✅ 芜湖衍生列补充完成 + 转移规模计算完成，全新完整表已保存！
最终完整表 CSV：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\各城市2026-2035算力规模_含芜湖修正+转移计算完整表.csv
最终完整表 Excel：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\各城市2026-2035算力规模_含芜湖修正+转移计算完整表.xlsx

【核心结果预览：2026年芜湖相关转移汇总】
                       城市  L0推理算力×30%(PFLOPS)  推理算力转移规模(PFLOPS)  L0推理算力×10%(PFLOPS)  10%推理转移规模(PFLOPS)
                  Beijing             1743.60              0.00              581.20               0.00
                 Shanghai             2699.90              0.00              899.97               0.00
                Guangzhou             1378.66              0.00              459.56               0.00
                 Shenzhen             1619.94              0.00              539.98               0.00
                  Chengdu                0.00           3488.70                0.00               0.00
                Chongqing                0.00           3488.70                0.00               0.00
              Zhangjiakou                0.00           1743.

In [21]:
import pandas as pd
import numpy as np
import os

# 你的专属保存路径
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
# 自动创建文件夹（不存在则新建）
os.makedirs(save_path, exist_ok=True)
# 数据结构：{城市名称: [省份, 2025因子, 2030因子, 2035因子]}
city_carbon_base = {
    "Zhangjiakou":               ["河北省",    0.714, 0.666, 0.512],
    "Hohhot (Helingeer)":        ["内蒙古",    0.791, 0.783, 0.665],
    "Zhongwei":                  ["宁夏",      0.703, 0.643, 0.452],
    "Wuhu":                      ["安徽省",    0.725, 0.650, 0.546],
    "Qingyang":                  ["甘肃省",    0.433, 0.391, 0.223],
    "Chongqing":                 ["重庆市",    0.231, 0.193, 0.131],
    "Chengdu":                   ["四川省",    0.103, 0.073, 0.040],
    "Guiyang (Gui'an New Area)": ["贵州省",    0.393, 0.260, 0.146],
    "Shaoguan":                  ["广东省",    0.359, 0.318, 0.269]
}

# 需要计算的目标年份：2026 - 2035
target_years = list(range(2026, 2036))

# 关键插值节点
y2025, f2025 = 2025, None
y2030, f2030 = 2030, None
y2035, f2035 = 2035, None
def linear_interp(year, f_2025, f_2030, f_2035):
    """
    分段线性插值计算逐年碳排放因子
    :param year: 待计算年份 (2026-2035)
    :param f_2025: 2025年基准因子
    :param f_2030: 2030年基准因子
    :param f_2035: 2035年基准因子
    :return: 当年碳排放因子（保留3位小数）
    """
    if 2025 < year <= 2030:
        # 第一段插值：2025 ~ 2030
        factor = f_2025 + (f_2030 - f_2025) * (year - 2025) / (2030 - 2025)
    else:
        # 第二段插值：2030 ~ 2035
        factor = f_2030 + (f_2035 - f_2030) * (year - 2030) / (2035 - 2030)
    return round(factor, 3)
    result_list = []

# 遍历每一个承接城市
for city, info in city_carbon_base.items():
    province, f25, f30, f35 = info
    # 遍历每一个目标年份 2026-2035
    for year in target_years:
        carbon_factor = linear_interp(year, f25, f30, f35)
        result_list.append({
            "城市": city,
            "所属省份": province,
            "年份": year,
            "电网碳排放因子": carbon_factor
        })

# 转为DataFrame数据表
df_factor = pd.DataFrame(result_list)

# 预览前20行数据
print("=== 2026-2035 各城市电网碳排放因子（插值结果）预览 ===")
print(df_factor.head(20).to_string(index=False))
# 定义文件名
csv_name = "九大城市2026-2035电网碳排放因子_线性插值.csv"
excel_name = "九大城市2026-2035电网碳排放因子_线性插值.xlsx"

csv_full_path = os.path.join(save_path, csv_name)
excel_full_path = os.path.join(save_path, excel_name)

# 保存文件
df_factor.to_csv(csv_full_path, index=False, encoding="utf-8-sig")
df_factor.to_excel(excel_path, index=False, engine="openpyxl")

print("\n✅ 插值计算完成，文件已保存！")
print(f"CSV 文件路径：{csv_full_path}")
print(f"Excel 文件路径：{excel_full_path}")

=== 2026-2035 各城市电网碳排放因子（插值结果）预览 ===
                城市 所属省份   年份  电网碳排放因子
       Zhangjiakou  河北省 2026    0.704
       Zhangjiakou  河北省 2027    0.695
       Zhangjiakou  河北省 2028    0.685
       Zhangjiakou  河北省 2029    0.676
       Zhangjiakou  河北省 2030    0.666
       Zhangjiakou  河北省 2031    0.635
       Zhangjiakou  河北省 2032    0.604
       Zhangjiakou  河北省 2033    0.574
       Zhangjiakou  河北省 2034    0.543
       Zhangjiakou  河北省 2035    0.512
Hohhot (Helingeer)  内蒙古 2026    0.789
Hohhot (Helingeer)  内蒙古 2027    0.788
Hohhot (Helingeer)  内蒙古 2028    0.786
Hohhot (Helingeer)  内蒙古 2029    0.785
Hohhot (Helingeer)  内蒙古 2030    0.783
Hohhot (Helingeer)  内蒙古 2031    0.759
Hohhot (Helingeer)  内蒙古 2032    0.736
Hohhot (Helingeer)  内蒙古 2033    0.712
Hohhot (Helingeer)  内蒙古 2034    0.689
Hohhot (Helingeer)  内蒙古 2035    0.665

✅ 插值计算完成，文件已保存！
CSV 文件路径：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\九大城市2026-2035电网碳排放因子_线性插值.csv
Excel 文件路径：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\九大城市2026-2035电网碳排放因子_线性插值.xlsx

In [22]:
import pandas as pd
import numpy as np
import os

# 固定保存路径
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
os.makedirs(save_path, exist_ok=True)
# 2025年基准数据：{城市: [上架率(%), PUE]}
base_2025 = {
    "Beijing":                  [65, 1.20],
    "Shanghai":                 [55, 1.20],
    "Guangzhou":                [50, 1.20],
    "Shenzhen":                 [50, 1.20],
    "Chengdu":                  [65, 1.25],
    "Chongqing":                [65, 1.25],
    "Zhangjiakou":              [60, 1.25],
    "Wuhu":                     [50, 1.25],
    "Shaoguan":                 [45, 1.25],
    "Hohhot (Helingeer)":       [55, 1.15],
    "Guiyang (Gui'an New Area)":[55, 1.20],
    "Qingyang":                 [80, 1.20],
    "Zhongwei":                 [80, 1.18]
}

# 2030年统一目标值
target_util = 80    # 上架率目标(%)
target_pue = 1.15   # PUE目标

# 预测年份范围：2026 - 2035
year_list = list(range(2026, 2036))
city_list = list(base_2025.keys())
def linear_predict(year, val_2025, val_2030):
    """
    分段线性预测函数
    2025 < 年份 ≤2030：线性过渡
    年份 >2030：保持2030目标值不变
    """
    if 2025 < year <= 2030:
        delta_total = val_2030 - val_2025
        year_gap = year - 2025
        total_gap = 2030 - 2025
        res = val_2025 + delta_total * (year_gap / total_gap)
    else:
        res = val_2030
    return res
    result_data = []

for city in city_list:
    util_25, pue_25 = base_2025[city]
    for year in year_list:
        # 计算当年上架率（保留1位小数）
        util_current = round(linear_predict(year, util_25, target_util), 1)
        # 计算当年PUE（保留2位小数，行业通用精度）
        pue_current = round(linear_predict(year, pue_25, target_pue), 2)
        
        result_data.append({
            "城市": city,
            "年份": year,
            "上架率(%)": util_current,
            "PUE": pue_current
        })

# 转为数据表
df_result = pd.DataFrame(result_data)

# 预览前25行
print("===== 2026-2035 上架率 & PUE 预测结果预览 =====")
print(df_result.head(25).to_string(index=False))
# 文件名
csv_name = "各城市2026-2035_上架率_PUE预测.csv"
excel_name = "各城市2026-2035_上架率_PUE预测.xlsx"

csv_path = os.path.join(save_path, csv_name)
excel_path = os.path.join(save_path, excel_name)

# 保存
df_result.to_csv(csv_path, index=False, encoding="utf-8-sig")
df_result.to_excel(excel_path, index=False, engine="openpyxl")

print("\n✅ 数据保存完成！")
print(f"CSV路径：{csv_path}")
print(f"Excel路径：{excel_path}")

===== 2026-2035 上架率 & PUE 预测结果预览 =====
       城市 节点层级   年份  通用算力占比(%)  智能算力占比(%)  推理算力占比(智算内,%)  训练算力占比(智算内,%)  上架率(%)  PUE
  Beijing   L0 2026      55.48      44.52          59.34          40.66     NaN  NaN
  Beijing   L0 2027      48.66      51.34          61.38          38.62     NaN  NaN
  Beijing   L0 2028      41.24      58.76          63.62          36.38     NaN  NaN
  Beijing   L0 2029      34.42      65.58          65.66          34.34     NaN  NaN
  Beijing   L0 2030      20.00      80.00          70.00          30.00     NaN  NaN
  Beijing   L0 2031      20.00      80.00          70.00          30.00     NaN  NaN
  Beijing   L0 2032      20.00      80.00          70.00          30.00     NaN  NaN
  Beijing   L0 2033      20.00      80.00          70.00          30.00     NaN  NaN
  Beijing   L0 2034      20.00      80.00          70.00          30.00     NaN  NaN
  Beijing   L0 2035      20.00      80.00          70.00          30.00     NaN  NaN
 Shanghai   L0 2026      4

In [25]:
import pandas as pd
import numpy as np
import os
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

# 统一文件路径
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
os.makedirs(save_path, exist_ok=True)
# ---------------------- 30%推理算力转移规则（修正版） ----------------------
rule_infer_30 = {
    "Beijing": {"Zhangjiakou": 1.0},
    "Shanghai": {"Wuhu": 0.5, "Chengdu": 0.25, "Chongqing": 0.25},
    "Wuhu": {"Chengdu": 0.5, "Chongqing": 0.5},
    "Guangzhou": {"Shaoguan": 1.0},
    "Shenzhen": {"Shaoguan": 1.0}
}

# ---------------------- 100%训练算力转移规则（修正版，补全所有转出城市） ----------------------
rule_train_100 = {
    "Beijing": {"Hohhot (Helingeer)": 0.6, "Zhongwei": 0.4},
    "Zhangjiakou": {"Hohhot (Helingeer)": 0.6, "Zhongwei": 0.4},
    "Shanghai": {"Qingyang": 1.0},
    "Wuhu": {"Qingyang": 1.0},
    "Chengdu": {"Qingyang": 0.5, "Guiyang (Gui'an New Area)": 0.5},
    "Chongqing": {"Qingyang": 0.5, "Guiyang (Gui'an New Area)": 0.5},
    "Guangzhou": {"Guiyang (Gui'an New Area)": 1.0},
    "Shenzhen": {"Guiyang (Gui'an New Area)": 1.0},
    "Shaoguan": {"Guiyang (Gui'an New Area)": 1.0}
}

# 所有年份范围
year_list = list(range(2026, 2036))
# 读取你之前生成的完整算力主表
df_source = pd.read_excel(
    os.path.join(save_path, "各城市2026-2035算力规模_含芜湖修正+转移计算完整表.xlsx")
)

# 只保留需要的核心列，修正转出基数
df_core = df_source[[
    "城市", "年份",
    "推理算力(PFLOPS)",  # 修正：总推理算力，作为30%转移的基数
    "训练算力(PFLOPS)"   # 修正：总训练算力，作为100%转移的基数
]].copy()

print("=== 原始数据读取完成，前15行预览 ===")
print(df_core.head(15).to_string(index=False))
# 初始化结果容器
result_total = []  # 带年份的总表
result_by_year = {}  # 分年份的表，key=年份，value=DataFrame

# 按年份循环处理
for year in year_list:
    # 筛选当前年份的数据
    df_year = df_core[df_core["年份"] == year].copy()
    year_result = []
    
    # ---------------------- 处理30%推理算力转移 ----------------------
    for source_city, target_rules in rule_infer_30.items():
        # 获取当前城市当年的总推理算力，计算30%转出总规模
        total_infer = df_year[df_year["城市"] == source_city]["推理算力(PFLOPS)"].values[0]
        transfer_total = total_infer * 0.3
        # 按比例分配到每个目标城市
        for target_city, ratio in target_rules.items():
            transfer_value = round(transfer_total * ratio, 1)
            # 存入结果
            year_result.append({
                "Source": source_city,
                "Target": target_city,
                "Type": "推理",
                "Computing power (PFLOPS)": transfer_value
            })
            # 存入总表
            result_total.append({
                "年份": year,
                "Source": source_city,
                "Target": target_city,
                "Type": "推理",
                "Computing power (PFLOPS)": transfer_value
            })
    
    # ---------------------- 处理100%训练算力转移 ----------------------
    for source_city, target_rules in rule_train_100.items():
        # 获取当前城市当年的总训练算力，100%转出
        transfer_total = df_year[df_year["城市"] == source_city]["训练算力(PFLOPS)"].values[0]
        # 按比例分配到每个目标城市
        for target_city, ratio in target_rules.items():
            transfer_value = round(transfer_total * ratio, 1)
            # 存入结果
            year_result.append({
                "Source": source_city,
                "Target": target_city,
                "Type": "训练",
                "Computing power (PFLOPS)": transfer_value
            })
            # 存入总表
            result_total.append({
                "年份": year,
                "Source": source_city,
                "Target": target_city,
                "Type": "训练",
                "Computing power (PFLOPS)": transfer_value
            })
    
    # 转为DataFrame，存入分年份字典
    result_by_year[year] = pd.DataFrame(year_result)

# 总表转为DataFrame
df_total = pd.DataFrame(result_total)

print("=== 修正后溯源表生成完成，2026年数据预览（无零值） ===")
print(result_by_year[2026].to_string(index=False))
# ---------------------- 1. 保存分年份Excel文件（每个年份一个Sheet） ----------------------
excel_year_path = os.path.join(save_path, "2026-2035算力转移溯源表_分年份_修正版.xlsx")
wb = Workbook()
# 删除默认Sheet
wb.remove(wb.active)

# 每个年份创建独立Sheet
for year, df in result_by_year.items():
    ws = wb.create_sheet(title=f"{year}年")
    # 写入数据
    for r in dataframe_to_rows(df, index=False, header=True):
        ws.append(r)
    # 调整列宽
    for col in ws.columns:
        ws.column_dimensions[col[0].column_letter].width = 28

# 保存分年份Excel
wb.save(excel_year_path)

# ---------------------- 2. 保存总表（带年份） ----------------------
total_csv_path = os.path.join(save_path, "2026-2035算力转移溯源表_总表_修正版.csv")
total_excel_path = os.path.join(save_path, "2026-2035算力转移溯源表_总表_修正版.xlsx")

df_total.to_csv(total_csv_path, index=False, encoding="utf-8-sig")
df_total.to_excel(total_excel_path, index=False, engine="openpyxl")

print("✅ 修正后所有溯源表保存完成！")
print(f"分年份Excel文件：{excel_year_path}")
print(f"总表CSV文件：{total_csv_path}")
print(f"总表Excel文件：{total_excel_path}")

=== 原始数据读取完成，前15行预览 ===
      城市   年份  推理算力(PFLOPS)  训练算力(PFLOPS)
 Beijing 2026       5812.00       3982.40
 Beijing 2027       7626.02       4798.26
 Beijing 2028       9951.38       5690.53
 Beijing 2029      12608.78       6594.36
 Beijing 2030      18037.71       7730.45
 Beijing 2031      18759.22       8039.67
 Beijing 2032      19509.59       8361.25
 Beijing 2033      20289.98       8695.70
 Beijing 2034      21101.57       9043.53
 Beijing 2035      21945.64       9405.27
Shanghai 2026       8999.68       7134.57
Shanghai 2027      11109.83       7878.09
Shanghai 2028      13755.79       8614.99
Shanghai 2029      16766.92       9341.64
Shanghai 2030      22547.14       9663.06
=== 修正后溯源表生成完成，2026年数据预览（无零值） ===
     Source                    Target Type  Computing power (PFLOPS)
    Beijing               Zhangjiakou   推理                    1743.6
   Shanghai                      Wuhu   推理                    1350.0
   Shanghai                   Chengdu   推理                     

In [27]:
import pandas as pd
import numpy as np
import os
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

# 统一文件路径（和你之前的路径完全一致）
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
os.makedirs(save_path, exist_ok=True)

# 公式固定常数：2*365*24*10^(-12)
CONST = 2 * 365 * 24 * (10 ** -12)
print(f"公式固定常数 = {CONST:.10e}")
# ---------------------- 1. 芯片参数定义（和之前完全一致） ----------------------
chip_param = {
    "2026_2030": {
        "推理": (60, 65),
        "训练": (67, 700)
    },
    "2031_2035": {
        "推理": (105, 55),
        "训练": (1152, 1000)
    }
}

def get_chip_params(year, power_type):
    """根据年份+算力类型，返回芯片算力、热设计功率"""
    if 2026 <= year <= 2030:
        return chip_param["2026_2030"][power_type]
    else:
        return chip_param["2031_2035"][power_type]

# ---------------------- 2. 读取基础数据表（核心修正） ----------------------
# 读取修正后的算力转移总表（Source、Target、Type、算力规模、年份）
df_transfer = pd.read_excel(
    os.path.join(save_path, "2026-2035算力转移溯源表_总表_修正版.xlsx")
)

# 读取【纯PUE/上架率表】，确保无冗余列、无NaN
# 该表仅包含4列：城市、年份、上架率(%)、PUE
df_pue_util = pd.read_excel(
    os.path.join(save_path, "各城市2026-2035_上架率_PUE预测.xlsx")
)[["城市", "年份", "上架率(%)", "PUE"]].copy()

# 清洗数据：去除NaN行，确保所有城市-年份都有参数
df_pue_util = df_pue_util.dropna(subset=["上架率(%)", "PUE"]).reset_index(drop=True)

# 预览数据，确认无NaN
print("=== 清洗后的PUE/上架率表前15行预览（无NaN） ===")
print(df_pue_util.head(15).to_string(index=False))
print(f"\n总参数行数：{len(df_pue_util)}")
print(f"覆盖城市数量：{df_pue_util['城市'].nunique()}")
print(f"覆盖年份范围：{df_pue_util['年份'].min()} ~ {df_pue_util['年份'].max()}")
# ---------------------- 1. 合并数据：按Target城市+年份匹配PUE/上架率（核心修正） ----------------------
# 列名重命名，确保匹配：Target城市 → 城市
df_merge = pd.merge(
    df_transfer,
    df_pue_util.rename(columns={"城市": "Target"}),
    on=["Target", "年份"],
    how="left"
)

# 检查无匹配的行（确保所有Target城市都有参数）
print(f"\n总转移线路行数：{len(df_merge)}")
print(f"无匹配PUE/上架率的线路数：{len(df_merge[df_merge['PUE'].isna()])}")
# 打印无匹配的城市，方便排查
if len(df_merge[df_merge['PUE'].isna()]) > 0:
    print("无匹配的Target城市：")
    print(df_merge[df_merge['PUE'].isna()]["Target"].unique())

# ---------------------- 2. 定义能源计算函数（使用Target城市的参数） ----------------------
def calc_energy(row):
    """
    单条线路能源计算函数（核心修正：使用Target城市的上架率、PUE）
    输入：DataFrame行数据
    输出：年能源消耗（TWh），保留8位小数
    """
    # 提取参数
    year = row["年份"]
    power_type = row["Type"]
    trans_flops = row["Computing power (PFLOPS)"]
    # 核心修正：使用Target城市的上架率、PUE
    util_rate = row["上架率(%)"]
    pue = row["PUE"]
    
    # 跳过无参数的行（如果有）
    if pd.isna(util_rate) or pd.isna(pue):
        return np.nan
    
    # 获取芯片参数
    chip_flops, power_w = get_chip_params(year, power_type)
    
    # 核心公式计算
    energy = (
        trans_flops
        * 1000
        / chip_flops
        * power_w
        * CONST
        * (util_rate / 100)
        * pue
    )
    
    # 保留8位小数，和你示例格式一致
    return round(energy, 8)

# ---------------------- 3. 批量计算能源消耗 ----------------------
df_merge["Energy (TWh)"] = df_merge.apply(calc_energy, axis=1)

# 生成最终的能源转移表结构
df_energy_final = df_merge[[
    "年份", "Source", "Target", "Type",
    "Computing power (PFLOPS)", "Energy (TWh)"
]].copy()

# 去除Energy为NaN的行（如果有）
df_energy_final = df_energy_final.dropna(subset=["Energy (TWh)"]).reset_index(drop=True)

print("\n=== 修正后能源转移表前15行预览（无NaN） ===")
print(df_energy_final.head(15).to_string(index=False))
# ---------------------- 1. 生成分年份Excel文件（每个年份一个Sheet） ----------------------
excel_year_path = os.path.join(save_path, "2026-2035能源转移溯源表_分年份_修正版.xlsx")
wb = Workbook()
# 删除默认Sheet
wb.remove(wb.active)

# 按年份分组，每个年份创建独立Sheet
year_list = sorted(df_energy_final["年份"].unique())
for year in year_list:
    df_year = df_energy_final[df_energy_final["年份"] == year].drop(columns=["年份"])
    ws = wb.create_sheet(title=f"{year}年")
    # 写入数据
    for r in dataframe_to_rows(df_year, index=False, header=True):
        ws.append(r)
    # 调整列宽
    for col in ws.columns:
        ws.column_dimensions[col[0].column_letter].width = 28

# 保存分年份Excel
wb.save(excel_year_path)

# ---------------------- 2. 保存总表（带年份） ----------------------
total_csv_path = os.path.join(save_path, "2026-2035能源转移溯源表_总表_修正版.csv")
total_excel_path = os.path.join(save_path, "2026-2035能源转移溯源表_总表_修正版.xlsx")

df_energy_final.to_csv(total_csv_path, index=False, encoding="utf-8-sig")
df_energy_final.to_excel(total_excel_path, index=False, engine="openpyxl")

print("✅ 修正后所有能源转移表生成完成！")
print(f"分年份Excel文件：{excel_year_path}")
print(f"总表CSV文件：{total_csv_path}")
print(f"总表Excel文件：{total_excel_path}")


公式固定常数 = 1.7520000000e-08
=== 清洗后的PUE/上架率表前15行预览（无NaN） ===
      城市   年份  上架率(%)  PUE
 Beijing 2026    68.0 1.19
 Beijing 2027    71.0 1.18
 Beijing 2028    74.0 1.17
 Beijing 2029    77.0 1.16
 Beijing 2030    80.0 1.15
 Beijing 2031    80.0 1.15
 Beijing 2032    80.0 1.15
 Beijing 2033    80.0 1.15
 Beijing 2034    80.0 1.15
 Beijing 2035    80.0 1.15
Shanghai 2026    60.0 1.19
Shanghai 2027    65.0 1.18
Shanghai 2028    70.0 1.17
Shanghai 2029    75.0 1.16
Shanghai 2030    80.0 1.15

总参数行数：130
覆盖城市数量：13
覆盖年份范围：2026 ~ 2035

总转移线路行数：210
无匹配PUE/上架率的线路数：0

=== 修正后能源转移表前15行预览（无NaN） ===
  年份      Source             Target Type  Computing power (PFLOPS)  Energy (TWh)
2026     Beijing        Zhangjiakou   推理                    1743.6      0.026051
2026    Shanghai               Wuhu   推理                    1350.0      0.017649
2026    Shanghai            Chengdu   推理                     675.0      0.010716
2026    Shanghai          Chongqing   推理                     675.0      0.010716
2026

In [29]:
#情景二
import pandas as pd
import numpy as np
import os
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

# 统一文件路径
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
os.makedirs(save_path, exist_ok=True)

# 能耗公式固定常数：2 * 365 * 24 * 10^(-12)
CONST = 2 * 365 * 24 * (10 ** -12)
print(f"公式固定常数 = {CONST:.10e}")

# 目标年份范围
year_list = list(range(2026, 2036))
# 1. 50%推理 流向规则
rule_infer_50 = {
    "Beijing": {"Zhangjiakou": 1.0},
    "Shanghai": {"Wuhu": 0.5, "Chengdu": 0.25, "Chongqing": 0.25},
    "Wuhu": {"Chengdu": 0.5, "Chongqing": 0.5},
    "Guangzhou": {"Shaoguan": 1.0},
    "Shenzhen": {"Shaoguan": 1.0}
}

# 2. 10%推理 流向规则
rule_infer_10 = {
    "Beijing": {"Hohhot (Helingeer)": 0.6, "Zhongwei": 0.4},
    "Shanghai": {"Qingyang": 1.0},
    "Wuhu": {"Qingyang": 1.0},
    "Guangzhou": {"Guiyang (Gui'an New Area)": 1.0},
    "Shenzhen": {"Guiyang (Gui'an New Area)": 1.0}
}

# 3. 100%训练 流向规则
rule_train_100 = {
    "Beijing": {"Hohhot (Helingeer)": 0.6, "Zhongwei": 0.4},
    "Zhangjiakou": {"Hohhot (Helingeer)": 0.6, "Zhongwei": 0.4},
    "Shanghai": {"Qingyang": 1.0},
    "Wuhu": {"Qingyang": 1.0},
    "Chengdu": {"Qingyang": 0.5, "Guiyang (Gui'an New Area)": 0.5},
    "Chongqing": {"Qingyang": 0.5, "Guiyang (Gui'an New Area)": 0.5},
    "Guangzhou": {"Guiyang (Gui'an New Area)": 1.0},
    "Shenzhen": {"Guiyang (Gui'an New Area)": 1.0},
    "Shaoguan": {"Guiyang (Gui'an New Area)": 1.0}
}
# 分时段芯片参数（2026-2030 / 2031-2035）
chip_param = {
    "2026_2030": {
        "推理": (60, 65),
        "训练": (67, 700)
    },
    "2031_2035": {
        "推理": (105, 55),
        "训练": (1152, 1000)
    }
}

def get_chip_params(year, power_type):
    """根据年份、算力类型返回：芯片算力(TFLOPS)、热设计功率(W)"""
    if 2026 <= year <= 2030:
        return chip_param["2026_2030"][power_type]
    else:
        return chip_param["2031_2035"][power_type]
        # 读取城市年度推理/训练总算力（基数来源）
df_base_power = pd.read_excel(
    os.path.join(save_path, "各城市2026-2035算力规模_含芜湖修正+转移计算完整表.xlsx")
)[["城市", "年份", "推理算力(PFLOPS)", "训练算力(PFLOPS)"]]

# 读取流入城市用的上架率、PUE表
df_pue_util = pd.read_excel(
    os.path.join(save_path, "各城市2026-2035_上架率_PUE预测.xlsx")
)[["城市", "年份", "上架率(%)", "PUE"]].dropna()

print("=== 基础算力表预览 ===")
print(df_base_power.head(10).to_string(index=False))
total_transfer_list = []    # 带年份总表
year_transfer_dict = {}     # 分年份数据

for year in year_list:
    df_year_power = df_base_power[df_base_power["年份"] == year].copy()
    year_result = []

    # ---------- 1. 50%推理算力转移 ----------
    for source_city, target_rules in rule_infer_50.items():
        infer_total = df_year_power[df_year_power["城市"] == source_city]["推理算力(PFLOPS)"].values[0]
        trans_total = infer_total * 0.5
        for target_city, ratio in target_rules.items():
            trans_val = round(trans_total * ratio, 1)
            row = {
                "年份": year, "Source": source_city, "Target": target_city,
                "Type": "推理", "Computing power (PFLOPS)": trans_val
            }
            year_result.append(row)
            total_transfer_list.append(row)

    # ---------- 2. 10%推理算力转移 ----------
    for source_city, target_rules in rule_infer_10.items():
        infer_total = df_year_power[df_year_power["城市"] == source_city]["推理算力(PFLOPS)"].values[0]
        trans_total = infer_total * 0.1
        for target_city, ratio in target_rules.items():
            trans_val = round(trans_total * ratio, 1)
            row = {
                "年份": year, "Source": source_city, "Target": target_city,
                "Type": "推理", "Computing power (PFLOPS)": trans_val
            }
            year_result.append(row)
            total_transfer_list.append(row)

    # ---------- 3. 100%训练算力转移 ----------
    for source_city, target_rules in rule_train_100.items():
        train_total = df_year_power[df_year_power["城市"] == source_city]["训练算力(PFLOPS)"].values[0]
        trans_total = train_total * 1.0
        for target_city, ratio in target_rules.items():
            trans_val = round(trans_total * ratio, 1)
            row = {
                "年份": year, "Source": source_city, "Target": target_city,
                "Type": "训练", "Computing power (PFLOPS)": trans_val
            }
            year_result.append(row)
            total_transfer_list.append(row)

    year_transfer_dict[year] = pd.DataFrame(year_result)

df_transfer_total = pd.DataFrame(total_transfer_list)
print("\n=== 2026年转移算力表预览 ===")
print(year_transfer_dict[2026].head(12).to_string(index=False))
# 按【Target(流入城市)+年份】匹配上架率、PUE
df_merge = pd.merge(
    df_transfer_total,
    df_pue_util.rename(columns={"城市": "Target"}),
    on=["Target", "年份"],
    how="left"
)

# 校验匹配完整性
print(f"\n总转移线路数：{len(df_merge)}")
print(f"无参数匹配线路数：{len(df_merge[df_merge['PUE'].isna()])}")

# 能耗计算函数
def calc_energy(row):
    year = row["年份"]
    p_type = row["Type"]
    trans_pf = row["Computing power (PFLOPS)"]
    util = row["上架率(%)"]
    pue = row["PUE"]

    if pd.isna(util) or pd.isna(pue):
        return np.nan

    chip_flops, chip_power = get_chip_params(year, p_type)
    energy = trans_pf * 1000 / chip_flops * chip_power * CONST * (util / 100) * pue
    return round(energy, 8)

# 批量计算能耗
df_merge["Energy (TWh)"] = df_merge.apply(calc_energy, axis=1)

# 最终输出表结构
df_final = df_merge[[
    "年份", "Source", "Target", "Type",
    "Computing power (PFLOPS)", "Energy (TWh)"
]].dropna(subset=["Energy (TWh)"]).reset_index(drop=True)

print("\n=== 2026年算力+能耗预览 ===")
print(df_final[df_final["年份"] == 2026].head(12).to_string(index=False))
# 分年份Excel文件
excel_year_path = os.path.join(
    save_path,
    "情景2_50%推理+10%推理+100%训练_2026-2035算力能耗_分年份.xlsx"
)
wb = Workbook()
wb.remove(wb.active)

for year in sorted(year_list):
    df_y = df_final[df_final["年份"] == year].drop(columns=["年份"])
    ws = wb.create_sheet(title=f"{year}年")
    for r in dataframe_to_rows(df_y, index=False, header=True):
        ws.append(r)
    # 调整列宽
    for col in ws.columns:
        ws.column_dimensions[col[0].column_letter].width = 30

wb.save(excel_year_path)

# 总表 CSV / Excel
total_csv = os.path.join(
    save_path,
    "情景2_50%推理+10%推理+100%训练_2026-2035算力能耗_总表.csv"
)
total_excel = os.path.join(
    save_path,
    "情景2_50%推理+10%推理+100%训练_2026-2035算力能耗_总表.xlsx"
)

df_final.to_csv(total_csv, index=False, encoding="utf-8-sig")
df_final.to_excel(total_excel, index=False, engine="openpyxl")

print("\n✅ 情景三 所有文件保存完成！")
print(f"分年份Excel：{excel_year_path}")
print(f"总表CSV：{total_csv}")
print(f"总表Excel：{total_excel}")


公式固定常数 = 1.7520000000e-08
=== 基础算力表预览 ===
     城市   年份  推理算力(PFLOPS)  训练算力(PFLOPS)
Beijing 2026       5812.00       3982.40
Beijing 2027       7626.02       4798.26
Beijing 2028       9951.38       5690.53
Beijing 2029      12608.78       6594.36
Beijing 2030      18037.71       7730.45
Beijing 2031      18759.22       8039.67
Beijing 2032      19509.59       8361.25
Beijing 2033      20289.98       8695.70
Beijing 2034      21101.57       9043.53
Beijing 2035      21945.64       9405.27

=== 2026年转移算力表预览 ===
  年份    Source             Target Type  Computing power (PFLOPS)
2026   Beijing        Zhangjiakou   推理                    2906.0
2026  Shanghai               Wuhu   推理                    2249.9
2026  Shanghai            Chengdu   推理                    1125.0
2026  Shanghai          Chongqing   推理                    1125.0
2026      Wuhu            Chengdu   推理                    4689.5
2026      Wuhu          Chongqing   推理                    4689.5
2026 Guangzhou           Shaog

In [42]:
import pandas as pd
import numpy as np
import os
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

# ===================== 1. 路径 & 基础常量 =====================
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
os.makedirs(save_path, exist_ok=True)

# 能耗公式固定常数 2*365*24*10^(-12)
CONST = 2 * 365 * (24 * (10 ** -12))
print(f"公式固定常数 CONST = {CONST:.10e}")

# 文件路径（统一使用清洗后的有效PUE表）
base_power_path = os.path.join(save_path, "各城市2026-2035算力规模_含芜湖修正+转移计算完整表.xlsx")
pue_valid_path = os.path.join(save_path, "各城市2026-2035_上架率_PUE_有效数据.xlsx")
carbon_factor_path = os.path.join(save_path, "九大城市2026-2035电网碳排放因子_线性插值.csv")

year_range = list(range(2026, 2036))

# ===================== 2. 城市名称标准化（和情景二完全一致） =====================
name_map = {
    "Hohhot (Helingeer)": "Hohhot",
    "Guiyang (Gui'an New Area)": "Guiyang"
}

def simplify_city_name(s):
    if pd.isna(s):
        return s
    s = str(s).strip()
    return name_map.get(s, s)

# ===================== 3. 情景一 转移规则：30%推理 + 100%训练 =====================
# 30%推理 流向规则
rule_infer_30 = {
    "Beijing": {"Zhangjiakou": 1.0},
    "Shanghai": {"Wuhu": 0.5, "Chengdu": 0.25, "Chongqing": 0.25},
    "Wuhu": {"Chengdu": 0.5, "Chongqing": 0.5},
    "Guangzhou": {"Shaoguan": 1.0},
    "Shenzhen": {"Shaoguan": 1.0}
}

# 100%训练 流向规则
rule_train_100 = {
    "Beijing": {"Hohhot (Helingeer)": 0.6, "Zhongwei": 0.4},
    "Zhangjiakou": {"Hohhot (Helingeer)": 0.6, "Zhongwei": 0.4},
    "Shanghai": {"Qingyang": 1.0},
    "Wuhu": {"Qingyang": 1.0},
    "Chengdu": {"Qingyang": 0.5, "Guiyang (Gui'an New Area)": 0.5},
    "Chongqing": {"Qingyang": 0.5, "Guiyang (Gui'an New Area)": 0.5},
    "Guangzhou": {"Guiyang (Gui'an New Area)": 1.0},
    "Shenzhen": {"Guiyang (Gui'an New Area)": 1.0},
    "Shaoguan": {"Guiyang (Gui'an New Area)": 1.0}
}

# ===================== 4. 芯片参数（两套情景共用） =====================
chip_param = {
    "2026_2030": {"推理": (60, 65), "训练": (67, 700)},
    "2031_2035": {"推理": (105, 55), "训练": (1152, 1000)}
}
def get_chip_params(year, power_type):
    if 2026 <= year <= 2030:
        return chip_param["2026_2030"][power_type]
    else:
        return chip_param["2031_2035"][power_type]

# ===================== 5. 读取并清洗基础数据 =====================
# 城市总算力表
df_base_power = pd.read_excel(base_power_path)[
    ["城市", "年份", "推理算力(PFLOPS)", "训练算力(PFLOPS)"]
].drop_duplicates().reset_index(drop=True)
df_base_power["城市"] = df_base_power["城市"].apply(simplify_city_name)

# 清洗后纯有效PUE/上架率表
df_pue_util = pd.read_excel(pue_valid_path)
df_pue_util["城市"] = df_pue_util["城市"].apply(simplify_city_name)
df_pue_util = df_pue_util.drop_duplicates(subset=["城市", "年份"]).reset_index(drop=True)
df_pue_util.rename(columns={"城市": "Target"}, inplace=True)

# 碳排放因子表
df_carbon = pd.read_csv(carbon_factor_path, encoding="utf-8-sig")[
    ["城市", "年份", "电网碳排放因子"]
]
df_carbon["城市"] = df_carbon["城市"].apply(simplify_city_name)
df_carbon = df_carbon.drop_duplicates(subset=["城市", "年份"]).reset_index(drop=True)
df_carbon.rename(
    columns={"城市": "Target", "电网碳排放因子": "emission factor (kgCO2/kWh)"},
    inplace=True
)

# ===================== 6. 生成情景一转移算力表（修复语法错误位置） =====================
transfer_total = []
year_transfer_dict = {}

for year in year_range:
    df_year_power = df_base_power[df_base_power["年份"] == year].copy()
    year_result = []

    # ---------- 30%推理算力 ----------
    for source, t_rule in rule_infer_30.items():
        infer_total = df_year_power["推理算力(PFLOPS)"].values[0]
        trans_base = infer_total * 0.3
        for target_raw, ratio in t_rule.items():
            target = simplify_city_name(target_raw)
            trans_val = round(trans_base * ratio, 1)
            year_result.append({
                "年份": year, "Source": source, "Target": target,
                "Type": "推理", "Computing power (PFLOPS)": trans_val
            })
            transfer_total.append({
                "年份": year, "Source": source, "Target": target,
                "Type": "推理", "Computing power (PFLOPS)": trans_val
            })

    # ---------- 100%训练算力 【重点修复此处语法错误】 ----------
    for source, t_rule in rule_train_100.items():
        # 原错误代码：train_total = df_year_power[df_year_power["训练算力(PFLOPS)"].values[0]
        # 修复后：直接取列 + 取值
        train_total = df_year_power["训练算力(PFLOPS)"].values[0]
        trans_base = train_total * 1.0
        for target_raw, ratio in t_rule.items():
            target = simplify_city_name(target_raw)
            trans_val = round(trans_base * ratio, 1)
            year_result.append({
                "年份": year, "Source": source, "Target": target,
                "Type": "训练", "Computing power (PFLOPS)": trans_val
            })
            transfer_total.append({
                "年份": year, "Source": source, "Target": target,
                "Type": "训练", "Computing power (PFLOPS)": trans_val
            })

    year_transfer_dict[year] = pd.DataFrame(year_result)

df_transfer = pd.DataFrame(transfer_total)
df_transfer = df_transfer.drop_duplicates().reset_index(drop=True)
print(f"情景一 转移算力总表行数：{len(df_transfer)}")

# ===================== 7. 合并参数 + 空值校验 =====================
df_merge = pd.merge(df_transfer, df_pue_util, on=["Target", "年份"], how="left")
df_merge = pd.merge(df_merge, df_carbon, on=["Target", "年份"], how="left")

print(f"上架率(%) 缺失行数：{df_merge['上架率(%)'].isna().sum()}")
print(f"PUE 缺失行数：{df_merge['PUE'].isna().sum()}")
print(f"碳排放因子 缺失行数：{df_merge['emission factor (kgCO2/kWh)'].isna().sum()}")

# 兜底填充
df_merge["上架率(%)"] = df_merge.groupby("Target")["上架率(%)"].transform(lambda x: x.fillna(x.mean()))
df_merge["PUE"] = df_merge.groupby("Target")["PUE"].transform(lambda x: x.fillna(x.mean()))
df_merge["emission factor (kgCO2/kWh)"] = df_merge.groupby("Target")["emission factor (kgCO2/kWh)"].transform(lambda x: x.fillna(x.mean()))

# ===================== 8. 能耗 / 热量 / 碳排放 计算 =====================
def calc_energy(row):
    year = row["年份"]
    p_type = row["Type"]
    trans_pf = row["Computing power (PFLOPS)"]
    util_rate = row["上架率(%)"]
    pue = row["PUE"]

    if pd.isna(util_rate) or pd.isna(pue) or pue <= 0:
        return 0.0
    chip_flops, chip_power = get_chip_params(year, p_type)
    energy = trans_pf * 1000 / chip_flops * chip_power * CONST * (util_rate / 100) * pue
    return round(energy, 8)

# 计算能耗(TWh)
df_merge["Energy (TWh)"] = df_merge.apply(calc_energy, axis=1)
# 计算热量(TJ)
df_merge["Heat (TJ)"] = np.where(
    df_merge["PUE"] <= 0,
    0.0,
    round((df_merge["Energy (TWh)"] / df_merge["PUE"]) * 3.6 * 1000, 7)
)
# 计算碳排放(Mt)
df_merge["Carbon emission (Mt)"] = round(
    df_merge["Energy (TWh)"] * df_merge["emission factor (kgCO2/kWh)"],
    9
)

# ===================== 9. 整理列顺序 =====================
final_cols = [
    "年份", "Source", "Target", "Type", "Computing power (PFLOPS)",
    "Energy (TWh)", "Heat (TJ)", "emission factor (kgCO2/kWh)", "Carbon emission (Mt)"
]
df_final = df_merge[final_cols].copy()
df_final = df_final.sort_values(by=["年份", "Source", "Target"]).reset_index(drop=True)
df_final = df_final.drop_duplicates().reset_index(drop=True)

# ===================== 10. 保存文件 =====================
# 分年份Excel
excel_year_path = os.path.join(
    save_path,
    "情景一_增量转移_算力_能耗_热量_碳排放_分年份_统一口径版.xlsx"
)
wb = Workbook()
wb.remove(wb.active)

for year in sorted(year_range):
    df_y = df_final[df_final["年份"] == year].drop(columns=["年份"])
    ws = wb.create_sheet(title=f"{year}年")
    for r in dataframe_to_rows(df_y, index=False):
        ws.append(r)
    for col in ws.columns:
        ws.column_dimensions[col[0].column_letter].width = 32

wb.save(excel_year_path)

# 总表 CSV + Excel
total_csv = os.path.join(
    save_path,
    "情景一_增量转移_算力_能耗_热量_碳排放_总表_统一口径版.csv"
)
total_excel = os.path.join(
    save_path,
    "情景一_增量转移_算力_能耗_热量_碳排放_总表_统一口径版.xlsx"
)
df_final.to_csv(total_csv, index=False, encoding="utf-8-sig")
df_final.to_excel(total_excel, index=False, engine="openpyxl")

# ===================== 11. 结果预览 =====================
print("\n✅ 情景一 统一口径版计算完成！")
print(f"分年份Excel：{excel_year_path}")
print(f"总表Excel：{total_excel}")

# 预览2026年数据
df_2026 = df_final[df_final["年份"] == 2026]
print("\n=== 2026年数据预览 ===")
print(df_2026.to_string(index=False))
print(f"2026年线路总数：{len(df_2026)}")

公式固定常数 CONST = 1.7520000000e-08
情景一 转移算力总表行数：210
上架率(%) 缺失行数：0
PUE 缺失行数：0
碳排放因子 缺失行数：0

✅ 情景一 统一口径版计算完成！
分年份Excel：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景一_增量转移_算力_能耗_热量_碳排放_分年份_统一口径版.xlsx
总表Excel：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景一_增量转移_算力_能耗_热量_碳排放_总表_统一口径版.xlsx

=== 2026年数据预览 ===
  年份      Source      Target Type  Computing power (PFLOPS)  Energy (TWh)   Heat (TJ)  emission factor (kgCO2/kWh)  Carbon emission (Mt)
2026     Beijing      Hohhot   训练                    2389.4      0.301783  944.713127                        0.789              0.238107
2026     Beijing Zhangjiakou   推理                    1743.6      0.026051   76.247502                        0.704              0.018340
2026     Beijing    Zhongwei   训练                    1593.0      0.272929  839.780154                        0.691              0.188594
2026     Chengdu     Guiyang   训练                    1991.2      0.260238  787.274108                        0.366              0.095247
2026     Chengdu    Qingyang   训练

In [38]:
import pandas as pd
import numpy as np
import os
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

# 你的文件路径
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
pue_file_path = os.path.join(save_path, "各城市2026-2035_上架率_PUE预测.xlsx")

# 读取Excel文件，自动识别有效数据
df_pue_raw = pd.read_excel(pue_file_path)

# 查看文件列名
print("Excel文件列名：", df_pue_raw.columns.tolist())

# 提取有效数据：上架率(%)和PUE非空的行
df_pue_valid = df_pue_raw.dropna(subset=["上架率(%)", "PUE"]).copy()
# 只保留需要的列
df_pue_valid = df_pue_valid[["城市", "年份", "上架率(%)", "PUE"]].reset_index(drop=True)

# 查看有效数据
print(f"有效上架率/PUE数据行数：{len(df_pue_valid)}")
print("有效数据预览：")
print(df_pue_valid.head(20).to_string(index=False))

# 保存清洗后的有效数据，方便后续使用
df_pue_valid.to_excel(os.path.join(save_path, "各城市2026-2035_上架率_PUE_有效数据.xlsx"), index=False)
print("\n✅ 清洗后的有效PUE/上架率数据已保存！")

Excel文件列名： ['城市', '节点层级', '年份', '通用算力占比(%)', '智能算力占比(%)', '推理算力占比(智算内,%)', '训练算力占比(智算内,%)', '上架率(%)', 'PUE']
有效上架率/PUE数据行数：130
有效数据预览：
      城市   年份  上架率(%)  PUE
 Beijing 2026    68.0 1.19
 Beijing 2027    71.0 1.18
 Beijing 2028    74.0 1.17
 Beijing 2029    77.0 1.16
 Beijing 2030    80.0 1.15
 Beijing 2031    80.0 1.15
 Beijing 2032    80.0 1.15
 Beijing 2033    80.0 1.15
 Beijing 2034    80.0 1.15
 Beijing 2035    80.0 1.15
Shanghai 2026    60.0 1.19
Shanghai 2027    65.0 1.18
Shanghai 2028    70.0 1.17
Shanghai 2029    75.0 1.16
Shanghai 2030    80.0 1.15
Shanghai 2031    80.0 1.15
Shanghai 2032    80.0 1.15
Shanghai 2033    80.0 1.15
Shanghai 2034    80.0 1.15
Shanghai 2035    80.0 1.15

✅ 清洗后的有效PUE/上架率数据已保存！


In [39]:
import pandas as pd
import numpy as np
import os
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

# ===================== 1. 路径 & 基础常量 =====================
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
os.makedirs(save_path, exist_ok=True)

# 能耗公式固定常数 2*365*24*10^(-12)
CONST = 2 * 365 * 24 * (10 ** -12)
print(f"公式固定常数 CONST = {CONST:.10e}")

# 文件路径
base_power_path = os.path.join(save_path, "各城市2026-2035算力规模_含芜湖修正+转移计算完整表.xlsx")
pue_valid_path = os.path.join(save_path, "各城市2026-2035_上架率_PUE_有效数据.xlsx")
carbon_factor_path = os.path.join(save_path, "九大城市2026-2035电网碳排放因子_线性插值.csv")

year_range = list(range(2026, 2036))

# ===================== 2. 城市名称映射（消除特殊符号匹配问题） =====================
name_map = {
    "Hohhot (Helingeer)": "Hohhot",
    "Guiyang (Gui'an New Area)": "Guiyang"
}

def simplify_city_name(s):
    if pd.isna(s):
        return s
    s = str(s).strip()
    return name_map.get(s, s)

# ===================== 3. 情景二 转移规则（50%推理 + 10%推理 + 100%训练） =====================
# 50%推理流向
rule_infer_50 = {
    "Beijing": {"Zhangjiakou": 1.0},
    "Shanghai": {"Wuhu": 0.5, "Chengdu": 0.25, "Chongqing": 0.25},
    "Wuhu": {"Chengdu": 0.5, "Chongqing": 0.5},
    "Guangzhou": {"Shaoguan": 1.0},
    "Shenzhen": {"Shaoguan": 1.0}
}
# 10%推理流向
rule_infer_10 = {
    "Beijing": {"Hohhot (Helingeer)": 0.6, "Zhongwei": 0.4},
    "Shanghai": {"Qingyang": 1.0},
    "Wuhu": {"Qingyang": 1.0},
    "Guangzhou": {"Guiyang (Gui'an New Area)": 1.0},
    "Shenzhen": {"Guiyang (Gui'an New Area)": 1.0}
}
# 100%训练流向
rule_train_100 = {
    "Beijing": {"Hohhot (Helingeer)": 0.6, "Zhongwei": 0.4},
    "Zhangjiakou": {"Hohhot (Helingeer)": 0.6, "Zhongwei": 0.4},
    "Shanghai": {"Qingyang": 1.0},
    "Wuhu": {"Qingyang": 1.0},
    "Chengdu": {"Qingyang": 0.5, "Guiyang (Gui'an New Area)": 0.5},
    "Chongqing": {"Qingyang": 0.5, "Guiyang (Gui'an New Area)": 0.5},
    "Guangzhou": {"Guiyang (Gui'an New Area)": 1.0},
    "Shenzhen": {"Guiyang (Gui'an New Area)": 1.0},
    "Shaoguan": {"Guiyang (Gui'an New Area)": 1.0}
}

# ===================== 4. 芯片参数 =====================
chip_param = {
    "2026_2030": {"推理": (60, 65), "训练": (67, 700)},
    "2031_2035": {"推理": (105, 55), "训练": (1152, 1000)}
}
def get_chip_params(year, power_type):
    if 2026 <= year <= 2030:
        return chip_param["2026_2030"][power_type]
    else:
        return chip_param["2031_2035"][power_type]

# ===================== 5. 读取所有基础数据（已清洗） =====================
# 5.1 城市算力表
df_base_power = pd.read_excel(base_power_path)[
    ["城市", "年份", "推理算力(PFLOPS)", "训练算力(PFLOPS)"]
].drop_duplicates().reset_index(drop=True)
df_base_power["城市"] = df_base_power["城市"].apply(simplify_city_name)

# 5.2 清洗后的PUE & 上架率表（核心修复：无空值）
df_pue_util = pd.read_excel(pue_valid_path)
df_pue_util["城市"] = df_pue_util["城市"].apply(simplify_city_name)
df_pue_util = df_pue_util.drop_duplicates(subset=["城市", "年份"]).reset_index(drop=True)
df_pue_util.rename(columns={"城市": "Target"}, inplace=True)

# 5.3 碳排放因子CSV
df_carbon = pd.read_csv(carbon_factor_path, encoding="utf-8-sig")[
    ["城市", "年份", "电网碳排放因子"]
]
df_carbon["城市"] = df_carbon["城市"].apply(simplify_city_name)
df_carbon = df_carbon.drop_duplicates(subset=["城市", "年份"]).reset_index(drop=True)
df_carbon.rename(
    columns={"城市": "Target", "电网碳排放因子": "emission factor (kgCO2/kWh)"},
    inplace=True
)

# ===================== 6. 生成转移算力表（27条/年，共270条） =====================
transfer_total = []
year_transfer_dict = {}

for year in year_range:
    df_year_power = df_base_power[df_base_power["年份"] == year].copy()
    year_result = []

    # 50%推理
    for source, t_rule in rule_infer_50.items():
        infer_total = df_year_power[df_year_power["城市"] == source]["推理算力(PFLOPS)"].values[0]
        trans_base = infer_total * 0.5
        for target_raw, ratio in t_rule.items():
            target = simplify_city_name(target_raw)
            trans_val = round(trans_base * ratio, 1)
            year_result.append({
                "年份": year, "Source": source, "Target": target,
                "Type": "推理", "Computing power (PFLOPS)": trans_val
            })
            transfer_total.append({
                "年份": year, "Source": source, "Target": target,
                "Type": "推理", "Computing power (PFLOPS)": trans_val
            })

    # 10%推理
    for source, t_rule in rule_infer_10.items():
        infer_total = df_year_power[df_year_power["城市"] == source]["推理算力(PFLOPS)"].values[0]
        trans_base = infer_total * 0.1
        for target_raw, ratio in t_rule.items():
            target = simplify_city_name(target_raw)
            trans_val = round(trans_base * ratio, 1)
            year_result.append({
                "年份": year, "Source": source, "Target": target,
                "Type": "推理", "Computing power (PFLOPS)": trans_val
            })
            transfer_total.append({
                "年份": year, "Source": source, "Target": target,
                "Type": "推理", "Computing power (PFLOPS)": trans_val
            })

    # 100%训练
    for source, t_rule in rule_train_100.items():
        train_total = df_year_power[df_year_power["城市"] == source]["训练算力(PFLOPS)"].values[0]
        trans_base = train_total * 1.0
        for target_raw, ratio in t_rule.items():
            target = simplify_city_name(target_raw)
            trans_val = round(trans_base * ratio, 1)
            year_result.append({
                "年份": year, "Source": source, "Target": target,
                "Type": "训练", "Computing power (PFLOPS)": trans_val
            })
            transfer_total.append({
                "年份": year, "Source": source, "Target": target,
                "Type": "训练", "Computing power (PFLOPS)": trans_val
            })

    year_transfer_dict[year] = pd.DataFrame(year_result)

df_transfer = pd.DataFrame(transfer_total)
df_transfer = df_transfer.drop_duplicates().reset_index(drop=True)
print(f"转移算力总表行数：{len(df_transfer)}")

# ===================== 7. 合并参数（100%匹配成功） =====================
df_merge = pd.merge(df_transfer, df_pue_util, on=["Target", "年份"], how="left")
df_merge = pd.merge(df_merge, df_carbon, on=["Target", "年份"], how="left")

# 校验缺失值（核心指标：全部为0）
print(f"上架率(%) 缺失行数：{df_merge['上架率(%)'].isna().sum()}")
print(f"PUE 缺失行数：{df_merge['PUE'].isna().sum()}")
print(f"碳排放因子 缺失行数：{df_merge['emission factor (kgCO2/kWh)'].isna().sum()}")

# 兜底填充（极端情况防护）
df_merge["上架率(%)"] = df_merge.groupby("Target")["上架率(%)"].transform(lambda x: x.fillna(x.mean()))
df_merge["PUE"] = df_merge.groupby("Target")["PUE"].transform(lambda x: x.fillna(x.mean()))
df_merge["emission factor (kgCO2/kWh)"] = df_merge.groupby("Target")["emission factor (kgCO2/kWh)"].transform(lambda x: x.fillna(x.mean()))

# ===================== 8. 能耗 / 热量 / 碳排放 计算（完整公式） =====================
def calc_energy(row):
    year = row["年份"]
    p_type = row["Type"]
    trans_pf = row["Computing power (PFLOPS)"]
    util_rate = row["上架率(%)"]
    pue = row["PUE"]

    if pd.isna(util_rate) or pd.isna(pue) or pue <= 0:
        return 0.0
    chip_flops, chip_power = get_chip_params(year, p_type)
    energy = trans_pf * 1000 / chip_flops * chip_power * CONST * (util_rate / 100) * pue
    return round(energy, 8)

# 计算能耗 (TWh)
df_merge["Energy (TWh)"] = df_merge.apply(calc_energy, axis=1)

# 计算热量 (TJ) = (Energy / PUE) * 3.6 * 1000
df_merge["Heat (TJ)"] = np.where(
    df_merge["PUE"] <= 0,
    0.0,
    round((df_merge["Energy (TWh)"] / df_merge["PUE"]) * 3.6 * 1000, 7)
)

# 计算碳排放 (Mt) = Energy * 排放因子
df_merge["Carbon emission (Mt)"] = round(
    df_merge["Energy (TWh)"] * df_merge["emission factor (kgCO2/kWh)"],
    9
)

# ===================== 9. 整理输出列顺序（与示例完全一致） =====================
final_cols = [
    "年份", "Source", "Target", "Type", "Computing power (PFLOPS)",
    "Energy (TWh)", "Heat (TJ)", "emission factor (kgCO2/kWh)", "Carbon emission (Mt)"
]
df_final = df_merge[final_cols].copy()
df_final = df_final.sort_values(by=["年份", "Source", "Target"]).reset_index(drop=True)
df_final = df_final.drop_duplicates().reset_index(drop=True)

# ===================== 10. 保存文件 =====================
# 分年份Excel
excel_year_path = os.path.join(
    save_path,
    "情景二_增量转移_算力_能耗_热量_碳排放_分年份_最终版.xlsx"
)
wb = Workbook()
wb.remove(wb.active)

for year in sorted(year_range):
    df_y = df_final[df_final["年份"] == year].drop(columns=["年份"])
    ws = wb.create_sheet(title=f"{year}年")
    for r in dataframe_to_rows(df_y, index=False, header=True):
        ws.append(r)
    for col in ws.columns:
        ws.column_dimensions[col[0].column_letter].width = 32

wb.save(excel_year_path)

# 总表 CSV + Excel
total_csv = os.path.join(
    save_path,
    "情景二_增量转移_算力_能耗_热量_碳排放_总表_最终版.csv"
)
total_excel = os.path.join(
    save_path,
    "情景二_增量转移_算力_能耗_热量_碳排放_总表_最终版.xlsx"
)
df_final.to_csv(total_csv, index=False, encoding="utf-8-sig")
df_final.to_excel(total_excel, index=False, engine="openpyxl")

# ===================== 11. 结果预览（27条线路，无0、无NaN） =====================
print("\n✅ 情景二 计算完成！文件已保存")
print(f"分年份Excel：{excel_year_path}")
print(f"总表Excel：{total_excel}")

# 预览2026年数据
df_2026 = df_final[df_final["年份"] == 2026]
print("\n=== 2026年 27条线路完整预览（有效数值） ===")
print(df_2026.to_string(index=False))
print(f"\n2026年线路总数：{len(df_2026)}")

公式固定常数 CONST = 1.7520000000e-08
转移算力总表行数：270
上架率(%) 缺失行数：0
PUE 缺失行数：0
碳排放因子 缺失行数：0

✅ 情景二 计算完成！文件已保存
分年份Excel：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景二_增量转移_算力_能耗_热量_碳排放_分年份_最终版.xlsx
总表Excel：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景二_增量转移_算力_能耗_热量_碳排放_总表_最终版.xlsx

=== 2026年 27条线路完整预览（有效数值） ===
  年份      Source      Target Type  Computing power (PFLOPS)  Energy (TWh)   Heat (TJ)  emission factor (kgCO2/kWh)  Carbon emission (Mt)
2026     Beijing      Hohhot   推理                     348.7      0.004567   14.295569                        0.789              0.003603
2026     Beijing      Hohhot   训练                    2389.4      0.301783  944.713127                        0.789              0.238107
2026     Beijing Zhangjiakou   推理                    2906.0      0.043419  127.079151                        0.704              0.030567
2026     Beijing    Zhongwei   推理                     232.5      0.004130   12.709015                        0.691              0.002854
2026     Beijing    Zhongwei 

In [46]:
import pandas as pd
import numpy as np
import os
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

# ===================== 1. 基础配置 & 全局常量 =====================
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
os.makedirs(save_path, exist_ok=True)

# 能耗公式固定常数 2 * 365 * 24 * 10^(-12)
CONST = 2 * 365 * 24 * (10 ** -12)
print(f"公式固定常数 CONST = {CONST:.10e}")

# 年份范围
year_range = list(range(2026, 2036))

# ---------------- 城市分组 & 折算比例 ----------------
# L2 4城：总算力 * 70%，使用【训练芯片】
l2_cities = ["Hohhot", "Guiyang", "Qingyang", "Zhongwei"]
ratio_l2 = 0.7

# L1 5城：总算力 * 30%，使用【推理芯片】
l1_cities = ["Chengdu", "Chongqing", "Zhangjiakou", "Wuhu", "Shaoguan"]
ratio_l1 = 0.3

# 合并所有目标城市（最终仅保留这9个）
target_all_cities = l2_cities + l1_cities

# ---------------- 城市名称标准化 ----------------
name_map = {
    "Hohhot (Helingeer)": "Hohhot",
    "Guiyang (Gui'an New Area)": "Guiyang"
}

def simplify_city_name(s):
    if pd.isna(s):
        return s
    s = str(s).strip()
    return name_map.get(s, s)

# ---------------- 芯片参数函数 ----------------
def get_train_chip(year):
    """训练芯片：L2四城专用 (TFLOPS, W)"""
    if 2026 <= year <= 2030:
        return 67, 700
    else:
        return 1152, 1000

def get_infer_chip(year):
    """推理芯片：L1五城专用 (TFLOPS, W)"""
    if 2026 <= year <= 2030:
        return 60, 65
    else:
        return 105, 55

# ===================== 2. 读取并清洗基础数据表 =====================
# 2.1 城市算力表：计算总算力
df_power = pd.read_excel(os.path.join(save_path, "各城市2026-2035算力规模_含芜湖修正+转移计算完整表.xlsx"))
df_power["城市"] = df_power["城市"].apply(simplify_city_name)
df_power["总算力(PFLOPS)"] = df_power["推理算力(PFLOPS)"] + df_power["训练算力(PFLOPS)"]
df_power = df_power[["城市", "年份", "总算力(PFLOPS)"]].drop_duplicates().reset_index(drop=True)

# 2.2 上架率 + PUE 表
df_pue = pd.read_excel(os.path.join(save_path, "各城市2026-2035_上架率_PUE_有效数据.xlsx"))
df_pue["城市"] = df_pue["城市"].apply(simplify_city_name)
df_pue = df_pue[["城市", "年份", "上架率(%)", "PUE"]].drop_duplicates(subset=["城市", "年份"]).reset_index(drop=True)

# 2.3 电网碳排放因子
df_carbon = pd.read_csv(
    os.path.join(save_path, "九大城市2026-2035电网碳排放因子_线性插值.csv"),
    encoding="utf-8-sig"
)
df_carbon["城市"] = df_carbon["城市"].apply(simplify_city_name)
df_carbon = df_carbon.rename(columns={"电网碳排放因子": "emission factor (kgCO2/kWh)"})
df_carbon = df_carbon[["城市", "年份", "emission factor (kgCO2/kWh)"]].drop_duplicates(subset=["城市", "年份"]).reset_index(drop=True)

# ===================== 3. 多表合并 + 筛选目标城市 =====================
df_merge = pd.merge(df_power, df_pue, on=["城市", "年份"], how="left")
df_merge = pd.merge(df_merge, df_carbon, on=["城市", "年份"], how="left")

# 仅保留指定9个城市
df_merge = df_merge[df_merge["城市"].isin(target_all_cities)].reset_index(drop=True)

# 空值兜底填充
df_merge["上架率(%)"] = df_merge.groupby("城市")["上架率(%)"].transform(lambda x: x.fillna(x.mean()))
df_merge["PUE"] = df_merge.groupby("城市")["PUE"].transform(lambda x: x.fillna(x.mean()))
df_merge["emission factor (kgCO2/kWh)"] = df_merge.groupby("城市")["emission factor (kgCO2/kWh)"].transform(lambda x: x.fillna(x.mean()))

# 缺失值校验
print(f"上架率缺失行数：{df_merge['上架率(%)'].isna().sum()}")
print(f"PUE缺失行数：{df_merge['PUE'].isna().sum()}")
print(f"碳排放因子缺失行数：{df_merge['emission factor (kgCO2/kWh)'].isna().sum()}")

# ===================== 4. 算力折算 + 能耗/热量/碳排放计算 =====================
def calc_metrics(row):
    city = row["城市"]
    year = row["年份"]
    total_pf = row["总算力(PFLOPS)"]
    util = row["上架率(%)"]
    pue = row["PUE"]
    ef = row["emission factor (kgCO2/kWh)"]

    if pd.isna(util) or pd.isna(pue) or pue <= 0:
        return 0.0, 0.0, 0.0

    # 区分城市分组、算力比例、芯片类型
    if city in l2_cities:
        calc_pf = total_pf * ratio_l2
        chip_flops, chip_pw = get_train_chip(year)
    elif city in l1_cities:
        calc_pf = total_pf * ratio_l1
        chip_flops, chip_pw = get_infer_chip(year)
    else:
        return 0.0, 0.0

    # 1. 能耗 TWh
    energy = calc_pf * 1000 / chip_flops * chip_pw * CONST * (util / 100) * pue
    energy = round(energy, 8)

    # 2. 热量 TJ
    heat = (energy / pue) * 3.6 * 1000
    heat = round(heat, 7)

    # 3. 碳排放 Mt
    carbon = energy * ef
    carbon = round(carbon, 9)

    return energy, heat, carbon

# 批量计算
df_merge[["Energy (TWh)", "Heat (TJ)", "Carbon emission (Mt)"]] = df_merge.apply(
    lambda row: pd.Series(calc_metrics(row)), axis=1
)

# ===================== 5. 整理输出列顺序 =====================
final_cols = [
    "年份", "城市", "总算力(PFLOPS)",
    "Energy (TWh)", "Heat (TJ)", "emission factor (kgCO2/kWh)", "Carbon emission (Mt)"
]
df_final = df_merge[final_cols].copy()
df_final = df_final.sort_values(by=["年份", "城市"]).reset_index(drop=True)

# ===================== 6. 保存文件（修复列宽写法） =====================
# 分年份Excel
excel_year_path = os.path.join(
    save_path,
    "L2四城+L1五城_折算算力_能耗热量碳排放_分年份.xlsx"
)
wb = Workbook()
wb.remove(wb.active)

for year in sorted(year_range):
    df_y = df_final[df_final["年份"] == year].drop(columns=["年份"])
    ws = wb.create_sheet(title=f"{year}")
    # 写入数据
    for row in dataframe_to_rows(df_y, index=False, header=True):
        ws.append(row)
    # ========== 修复列宽：标准openpyxl写法 ==========
    # 对当前工作表所有列设置宽度为32
    for col_idx in range(1, ws.max_column + 1):
        ws.column_dimensions[ws.cell(row=1, column=col_idx).column_letter].width = 32

wb.save(excel_year_path)

# 总表 Excel + CSV
total_excel = os.path.join(
    save_path,
    "L2四城+L1五城_折算算力_能耗热量碳排放_总表.xlsx"
)
total_csv = os.path.join(
    save_path,
    "L2四城+L1五城_折算算力_能耗热量碳排放_总表.csv"
)

df_final.to_excel(total_excel, index=False, engine="openpyxl")
df_final.to_csv(total_csv, index=False, encoding="utf-8-sig")

# ===================== 7. 结果预览 =====================
print("\n✅ 计算完成！文件已全部保存")
print(f"分年份Excel：{excel_year_path}")
print(f"总表Excel：{total_excel}")
print(f"总表CSV：{total_csv}")

# 预览2026年数据
print("\n=== 2026年 9座城市数据预览 ===")
df_2026 = df_final[df_final["年份"] == 2026]
print(df_2026.to_string(index=False))

# 预览2031年（芯片切换节点）
print("\n=== 2031年 9座城市数据预览 ===")
df_2031 = df_final[df_final["年份"] == 2031]
print(df_2031.to_string(index=False))

公式固定常数 CONST = 1.7520000000e-08
上架率缺失行数：0
PUE缺失行数：0
碳排放因子缺失行数：0

✅ 计算完成！文件已全部保存
分年份Excel：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\L2四城+L1五城_折算算力_能耗热量碳排放_分年份.xlsx
总表Excel：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\L2四城+L1五城_折算算力_能耗热量碳排放_总表.xlsx
总表CSV：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\L2四城+L1五城_折算算力_能耗热量碳排放_总表.csv

=== 2026年 9座城市数据预览 ===
  年份          城市  总算力(PFLOPS)  Energy (TWh)    Heat (TJ)  emission factor (kgCO2/kWh)  Carbon emission (Mt)
2026     Chengdu     19075.15      0.090845   265.886839                        0.097              0.008812
2026   Chongqing     15300.12      0.072866   213.267015                        0.223              0.016249
2026     Guiyang    153528.98     14.045718 42491.248790                        0.366              5.140733
2026      Hohhot    187752.40     16.599307 51963.049190                        0.789             13.096854
2026    Qingyang    184234.78     22.473150 67985.999546                        0.425              9.551089
2026    Shaoguan     11079.04 

In [47]:
import pandas as pd
import numpy as np
import os
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

# ===================== 1. 基础配置 & 路径 =====================
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
os.makedirs(save_path, exist_ok=True)

# 年份范围
year_range = list(range(2026, 2036))

# 目标9个城市（仅保留这些城市的合并结果）
target_cities = ["Hohhot", "Guiyang", "Qingyang", "Zhongwei", "Chengdu", "Chongqing", "Zhangjiakou", "Wuhu", "Shaoguan"]

# ---------------- 城市名称标准化（统一所有表的名称） ----------------
name_map = {
    "Hohhot (Helingeer)": "Hohhot",
    "Guiyang (Gui'an New Area)": "Guiyang"
}

def simplify_city_name(s):
    if pd.isna(s):
        return s
    s = str(s).strip()
    return name_map.get(s, s)

# ===================== 2. 读取所有核心数据表 =====================
# 2.1 读取存量折算表（L2+L1 9个城市）
df_stock = pd.read_excel(os.path.join(save_path, "L2四城+L1五城_折算算力_能耗热量碳排放_总表.xlsx"))
df_stock["城市"] = df_stock["城市"].apply(simplify_city_name)
# 仅保留目标9个城市
df_stock = df_stock[df_stock["城市"].isin(target_cities)].reset_index(drop=True)
# 重命名存量字段，区分增量
df_stock.rename(columns={
    "Energy (TWh)": "Stock_Energy (TWh)",
    "Heat (TJ)": "Stock_Heat (TJ)",
    "Carbon emission (Mt)": "Stock_Carbon (Mt)"
}, inplace=True)
print(f"存量表有效行数：{len(df_stock)}")

# 2.2 读取情景一增量表
df_scenario1 = pd.read_excel(os.path.join(save_path, "情景一_增量转移_算力_能耗_热量_碳排放_总表_统一口径版.xlsx"))
df_scenario1["Target"] = df_scenario1["Target"].apply(simplify_city_name)
# 仅保留目标9个城市的增量
df_scenario1 = df_scenario1[df_scenario1["Target"].isin(target_cities)].reset_index(drop=True)
print(f"情景一增量表有效行数：{len(df_scenario1)}")

# 2.3 读取情景二增量表
df_scenario2 = pd.read_excel(os.path.join(save_path, "情景二_增量转移_算力_能耗_热量_碳排放_总表_最终版.xlsx"))
df_scenario2["Target"] = df_scenario2["Target"].apply(simplify_city_name)
# 仅保留目标9个城市的增量
df_scenario2 = df_scenario2[df_scenario2["Target"].isin(target_cities)].reset_index(drop=True)
print(f"情景二增量表有效行数：{len(df_scenario2)}")

# ===================== 3. 增量数据按【年份+城市】汇总 =====================
def aggregate_increment(df_increment, scenario_name):
    """按年份+Target汇总增量数据，返回汇总表"""
    df_agg = df_increment.groupby(["年份", "Target"]).agg({
        "Energy (TWh)": "sum",
        "Heat (TJ)": "sum",
        "Carbon emission (Mt)": "sum"
    }).reset_index()
    # 重命名增量字段
    df_agg.rename(columns={
        "Target": "城市",
        "Energy (TWh)": f"{scenario_name}_Increment_Energy (TWh)",
        "Heat (TJ)": f"{scenario_name}_Increment_Heat (TJ)",
        "Carbon emission (Mt)": f"{scenario_name}_Increment_Carbon (Mt)"
    }, inplace=True)
    return df_agg

# 汇总两个情景的增量
df_s1_agg = aggregate_increment(df_scenario1, "S1")
df_s2_agg = aggregate_increment(df_scenario2, "S2")
print(f"情景一增量汇总行数：{len(df_s1_agg)}")
print(f"情景二增量汇总行数：{len(df_s2_agg)}")

# ===================== 4. 存量+增量合并，生成最终总表 =====================
def merge_stock_increment(df_stock, df_increment_agg, scenario_name):
    """合并存量和增量，计算总负荷"""
    # 按年份+城市合并
    df_merge = pd.merge(
        df_stock,
        df_increment_agg,
        on=["年份", "城市"],
        how="left"
    )
    # 无增量的城市，增量填充为0
    increment_cols = [
        f"{scenario_name}_Increment_Energy (TWh)",
        f"{scenario_name}_Increment_Heat (TJ)",
        f"{scenario_name}_Increment_Carbon (Mt)"
    ]
    df_merge[increment_cols] = df_merge[increment_cols].fillna(0)

    # 计算总负荷
    df_merge[f"{scenario_name}_Total_Energy (TWh)"] = df_merge["Stock_Energy (TWh)"] + df_merge[f"{scenario_name}_Increment_Energy (TWh)"]
    df_merge[f"{scenario_name}_Total_Heat (TJ)"] = df_merge["Stock_Heat (TJ)"] + df_merge[f"{scenario_name}_Increment_Heat (TJ)"]
    df_merge[f"{scenario_name}_Total_Carbon (Mt)"] = df_merge["Stock_Carbon (Mt)"] + df_merge[f"{scenario_name}_Increment_Carbon (Mt)"]

    # 整理列顺序
    final_cols = [
        "年份", "城市", "总算力(PFLOPS)",
        "Stock_Energy (TWh)", f"{scenario_name}_Increment_Energy (TWh)", f"{scenario_name}_Total_Energy (TWh)",
        "Stock_Heat (TJ)", f"{scenario_name}_Increment_Heat (TJ)", f"{scenario_name}_Total_Heat (TJ)",
        "emission factor (kgCO2/kWh)", "Stock_Carbon (Mt)", f"{scenario_name}_Increment_Carbon (Mt)", f"{scenario_name}_Total_Carbon (Mt)"
    ]
    df_final = df_merge[final_cols].copy()
    df_final = df_final.sort_values(by=["年份", "城市"]).reset_index(drop=True)
    return df_final

# 生成两个情景的最终总表
df_s1_final = merge_stock_increment(df_stock, df_s1_agg, "S1")
df_s2_final = merge_stock_increment(df_stock, df_s2_agg, "S2")
print(f"情景一最终总表行数：{len(df_s1_final)}")
print(f"情景二最终总表行数：{len(df_s2_final)}")

# ===================== 5. 保存文件（分年份Excel + 总表） =====================
def save_to_excel(df_final, scenario_name):
    """保存为分年份Excel和总表"""
    # 分年份Excel
    excel_year_path = os.path.join(
        save_path,
        f"{scenario_name}_存量+增量_总负荷_分年份.xlsx"
    )
    wb = Workbook()
    wb.remove(wb.active)

    for year in sorted(year_range):
        df_y = df_final[df_final["年份"] == year].drop(columns=["年份"])
        ws = wb.create_sheet(title=f"{year}年")
        for row in dataframe_to_rows(df_y, index=False, header=True):
            ws.append(row)
        # 设置列宽
        for col_idx in range(1, ws.max_column + 1):
            ws.column_dimensions[ws.cell(row=1, column=col_idx).column_letter].width = 32

    wb.save(excel_year_path)

    # 总表Excel + CSV
    total_excel = os.path.join(
        save_path,
        f"{scenario_name}_存量+增量_总负荷_总表.xlsx"
    )
    total_csv = os.path.join(
        save_path,
        f"{scenario_name}_存量+增量_总负荷_总表.csv"
    )
    df_final.to_excel(total_excel, index=False, engine="openpyxl")
    df_final.to_csv(total_csv, index=False, encoding="utf-8-sig")

    return excel_year_path, total_excel, total_csv

# 保存两个情景的文件
s1_files = save_to_excel(df_s1_final, "情景一")
s2_files = save_to_excel(df_s2_final, "情景二")

# ===================== 6. 结果预览 =====================
print("\n✅ 两个情景的存量+增量合并完成！文件已全部保存")
print("\n=== 情景一文件 ===")
print(f"分年份Excel：{s1_files[0]}")
print(f"总表Excel：{s1_files[1]}")
print(f"总表CSV：{s1_files[2]}")

print("\n=== 情景二文件 ===")
print(f"分年份Excel：{s2_files[0]}")
print(f"总表Excel：{s2_files[1]}")
print(f"总表CSV：{s2_files[2]}")

# 预览2026年情景一数据
print("\n=== 2026年 情景一 总负荷预览 ===")
df_s1_2026 = df_s1_final[df_s1_final["年份"] == 2026]
print(df_s1_2026[["年份", "城市", "Stock_Energy (TWh)", "S1_Increment_Energy (TWh)", "S1_Total_Energy (TWh)"]].to_string(index=False))

# 预览2026年情景二数据
print("\n=== 2026年 情景二 总负荷预览 ===")
df_s2_2026 = df_s2_final[df_s2_final["年份"] == 2026]
print(df_s2_2026[["年份", "城市", "Stock_Energy (TWh)", "S2_Increment_Energy (TWh)", "S2_Total_Energy (TWh)"]].to_string(index=False))

存量表有效行数：90
情景一增量表有效行数：210
情景二增量表有效行数：270
情景一增量汇总行数：90
情景二增量汇总行数：90
情景一最终总表行数：90
情景二最终总表行数：90

✅ 两个情景的存量+增量合并完成！文件已全部保存

=== 情景一文件 ===
分年份Excel：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景一_存量+增量_总负荷_分年份.xlsx
总表Excel：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景一_存量+增量_总负荷_总表.xlsx
总表CSV：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景一_存量+增量_总负荷_总表.csv

=== 情景二文件 ===
分年份Excel：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景二_存量+增量_总负荷_分年份.xlsx
总表Excel：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景二_存量+增量_总负荷_总表.xlsx
总表CSV：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景二_存量+增量_总负荷_总表.csv

=== 2026年 情景一 总负荷预览 ===
  年份          城市  Stock_Energy (TWh)  S1_Increment_Energy (TWh)  S1_Total_Energy (TWh)
2026     Chengdu            0.090845                   0.020760               0.111604
2026   Chongqing            0.072866                   0.020760               0.093626
2026     Guiyang           14.045718                   2.081903              16.127621
2026      Hohhot           16.599307                   0.603567              17.202874
2026    Q

In [49]:
import pandas as pd
import numpy as np
import os
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

# ===================== 1. 基础路径 & ERF参数配置 =====================
# 统一文件路径（沿用你之前的目录）
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
os.makedirs(save_path, exist_ok=True)

# 【三套余热方案 ERF 固定配置】训练/推理分开
erf_config = {
    # 方案1：低效率 训练40%，推理10%
    "方案1_可减少热量(TJ)": {"训练": 0.40, "推理": 0.10},
    # 方案2：中效率 训练65%，推理30%
    "方案2_可减少热量(TJ)": {"训练": 0.65, "推理": 0.30},
    # 方案3：高效率 训练90%，推理50%
    "方案3_可减少热量(TJ)": {"训练": 0.90, "推理": 0.50}
}

# 输入文件：情景一 增量转移总表（原始文件）
input_total = os.path.join(
    save_path,
    "情景一_增量转移_算力_能耗_热量_碳排放_总表_统一口径版.xlsx"
)

# 输出文件（新增余热列，文件名区分原文件）
out_total = os.path.join(
    save_path,
    "情景一_增量转移_附三套余热可减少热量_总表.xlsx"
)
out_year = os.path.join(
    save_path,
    "情景一_增量转移_附三套余热可减少热量_分年份.xlsx"
)

# ===================== 2. 读取原始数据 =====================
df = pd.read_excel(input_total)

# 校验核心字段（防止列名错误）
need_cols = ["Type", "Heat (TJ)"]
for col in need_cols:
    if col not in df.columns:
        raise Exception(f"原始表格缺失关键字段：{col}")

print(f"原始数据总行数：{len(df)}")
print("数据类型统计：")
print(df["Type"].value_counts())  # 查看训练/推理行数

# ===================== 3. 分类型计算三套方案【可减少热量】 =====================
# 循环遍历三套方案，逐列计算
for col_name, erf_dict in erf_config.items():
    train_erf = erf_dict["训练"]
    infer_erf = erf_dict["推理"]
    
    # 向量化判断类型并计算（训练用训练ERF，推理用推理ERF）
    df[col_name] = np.where(
        df["Type"] == "训练",
        df["Heat (TJ)"] * train_erf,
        df["Heat (TJ)"] * infer_erf
    )
    # 保留7位小数（和原Heat列精度保持一致）
    df[col_name] = df[col_name].round(7)

# ===================== 4. 整理列顺序（原始列在前，新增3列在最后） =====================
# 原始所有列 + 新增三列
new_cols = list(df.columns)
df = df[new_cols].sort_values(by=["年份", "Source", "Target"]).reset_index(drop=True)

# ===================== 5. 保存文件：总表 + 分年份Excel =====================
# 5.1 保存全局总表
df.to_excel(out_total, index=False, engine="openpyxl")
df.to_csv(out_total.replace(".xlsx", ".csv"), index=False, encoding="utf-8-sig")

# 5.2 生成分年份Excel（每年独立Sheet）
wb = Workbook()
wb.remove(wb.active)
year_list = sorted(df["年份"].unique())

for year in year_list:
    df_year = df[df["年份"] == year].drop(columns=["年份"])
    ws = wb.create_sheet(title=f"{year}年")
    # 写入数据
    for row in dataframe_to_rows(df_year, index=False, header=True):
        ws.append(row)
    # 统一列宽
    for col_idx in range(1, ws.max_column + 1):
        col_letter = ws.cell(row=1, column=col_idx).column_letter
        ws.column_dimensions[col_letter].width = 32

wb.save(out_year)

# ===================== 6. 结果预览 =====================
print("\n✅ 计算完成！已在原表追加三套方案【可减少热量】列")
print(f"更新后总表：{out_total}")
print(f"分年份表格：{out_year}")

# 预览2026年数据（核心字段校验）
print("\n===== 2026年数据预览（仅展示关键列）=====")
preview_cols = [
    "年份", "Source", "Target", "Type", "Heat (TJ)",
    "方案1_可减少热量(TJ)", "方案2_可减少热量(TJ)", "方案3_可减少热量(TJ)"
]
print(df[df["年份"] == 2026][preview_cols].to_string(index=False))

原始数据总行数：210
数据类型统计：
Type
训练    130
推理     80
Name: count, dtype: int64

✅ 计算完成！已在原表追加三套方案【可减少热量】列
更新后总表：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景一_增量转移_附三套余热可减少热量_总表.xlsx
分年份表格：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景一_增量转移_附三套余热可减少热量_分年份.xlsx

===== 2026年数据预览（仅展示关键列）=====
  年份      Source      Target Type   Heat (TJ)  方案1_可减少热量(TJ)  方案2_可减少热量(TJ)  方案3_可减少热量(TJ)
2026     Beijing      Hohhot   训练  944.713127     377.885251     614.063533     850.241814
2026     Beijing Zhangjiakou   推理   76.247502       7.624750      22.874251      38.123751
2026     Beijing    Zhongwei   训练  839.780154     335.912061     545.857100     755.802138
2026     Chengdu     Guiyang   训练  787.274108     314.909643     511.728170     708.546697
2026     Chengdu    Qingyang   训练 1049.698830     419.879532     682.304240     944.728947
2026   Chongqing     Guiyang   训练  787.274108     314.909643     511.728170     708.546697
2026   Chongqing    Qingyang   训练 1049.698830     419.879532     682.304240     944.728947
2026   

In [50]:
import pandas as pd
import numpy as np
import os
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

# ===================== 1. 基础路径 & ERF参数配置 =====================
# 统一文件路径（沿用你之前的目录）
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
os.makedirs(save_path, exist_ok=True)

# 【三套余热方案 ERF 固定配置】训练/推理分开
erf_config = {
    # 方案1：低效率 训练40%，推理10%
    "方案1_可减少热量(TJ)": {"训练": 0.40, "推理": 0.10},
    # 方案2：中效率 训练65%，推理30%
    "方案2_可减少热量(TJ)": {"训练": 0.65, "推理": 0.30},
    # 方案3：高效率 训练90%，推理50%
    "方案3_可减少热量(TJ)": {"训练": 0.90, "推理": 0.50}
}

# 输入文件：情景一 增量转移总表（原始文件）
input_total = os.path.join(
    save_path,
    "情景二_增量转移_算力_能耗_热量_碳排放_总表_最终版.xlsx"
)

# 输出文件（新增余热列，文件名区分原文件）
out_total = os.path.join(
    save_path,
    "情景二_增量转移_附三套余热可减少热量_总表.xlsx"
)
out_year = os.path.join(
    save_path,
    "情景二_增量转移_附三套余热可减少热量_分年份.xlsx"
)

# ===================== 2. 读取原始数据 =====================
df = pd.read_excel(input_total)

# 校验核心字段（防止列名错误）
need_cols = ["Type", "Heat (TJ)"]
for col in need_cols:
    if col not in df.columns:
        raise Exception(f"原始表格缺失关键字段：{col}")

print(f"原始数据总行数：{len(df)}")
print("数据类型统计：")
print(df["Type"].value_counts())  # 查看训练/推理行数

# ===================== 3. 分类型计算三套方案【可减少热量】 =====================
# 循环遍历三套方案，逐列计算
for col_name, erf_dict in erf_config.items():
    train_erf = erf_dict["训练"]
    infer_erf = erf_dict["推理"]
    
    # 向量化判断类型并计算（训练用训练ERF，推理用推理ERF）
    df[col_name] = np.where(
        df["Type"] == "训练",
        df["Heat (TJ)"] * train_erf,
        df["Heat (TJ)"] * infer_erf
    )
    # 保留7位小数（和原Heat列精度保持一致）
    df[col_name] = df[col_name].round(7)

# ===================== 4. 整理列顺序（原始列在前，新增3列在最后） =====================
# 原始所有列 + 新增三列
new_cols = list(df.columns)
df = df[new_cols].sort_values(by=["年份", "Source", "Target"]).reset_index(drop=True)

# ===================== 5. 保存文件：总表 + 分年份Excel =====================
# 5.1 保存全局总表
df.to_excel(out_total, index=False, engine="openpyxl")
df.to_csv(out_total.replace(".xlsx", ".csv"), index=False, encoding="utf-8-sig")

# 5.2 生成分年份Excel（每年独立Sheet）
wb = Workbook()
wb.remove(wb.active)
year_list = sorted(df["年份"].unique())

for year in year_list:
    df_year = df[df["年份"] == year].drop(columns=["年份"])
    ws = wb.create_sheet(title=f"{year}年")
    # 写入数据
    for row in dataframe_to_rows(df_year, index=False, header=True):
        ws.append(row)
    # 统一列宽
    for col_idx in range(1, ws.max_column + 1):
        col_letter = ws.cell(row=1, column=col_idx).column_letter
        ws.column_dimensions[col_letter].width = 32

wb.save(out_year)

# ===================== 6. 结果预览 =====================
print("\n✅ 计算完成！已在原表追加三套方案【可减少热量】列")
print(f"更新后总表：{out_total}")
print(f"分年份表格：{out_year}")

# 预览2026年数据（核心字段校验）
print("\n===== 2026年数据预览（仅展示关键列）=====")
preview_cols = [
    "年份", "Source", "Target", "Type", "Heat (TJ)",
    "方案1_可减少热量(TJ)", "方案2_可减少热量(TJ)", "方案3_可减少热量(TJ)"
]
print(df[df["年份"] == 2026][preview_cols].to_string(index=False))

原始数据总行数：270
数据类型统计：
Type
推理    140
训练    130
Name: count, dtype: int64

✅ 计算完成！已在原表追加三套方案【可减少热量】列
更新后总表：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景二_增量转移_附三套余热可减少热量_总表.xlsx
分年份表格：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景二_增量转移_附三套余热可减少热量_分年份.xlsx

===== 2026年数据预览（仅展示关键列）=====
  年份      Source      Target Type   Heat (TJ)  方案1_可减少热量(TJ)  方案2_可减少热量(TJ)  方案3_可减少热量(TJ)
2026     Beijing      Hohhot   推理   14.295569       1.429557       4.288671       7.147784
2026     Beijing      Hohhot   训练  944.713127     377.885251     614.063533     850.241814
2026     Beijing Zhangjiakou   推理  127.079151      12.707915      38.123745      63.539576
2026     Beijing    Zhongwei   推理   12.709015       1.270901       3.812705       6.354508
2026     Beijing    Zhongwei   训练  839.780154     335.912061     545.857100     755.802138
2026     Chengdu     Guiyang   训练 2387.759052     955.103621    1552.043384    2148.983147
2026     Chengdu    Qingyang   训练 3183.678756    1273.471503    2069.391192    2865.310881
2026   

In [51]:
import pandas as pd
import numpy as np
import os
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter

# ===================== 1. 基础配置 & 核心参数 =====================
# 统一文件路径
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
os.makedirs(save_path, exist_ok=True)

# 输入文件：情景一 存量+增量 分年份Excel
input_file = os.path.join(
    save_path,
    "情景一_存量+增量_总负荷_分年份.xlsx"
)

# 输出文件：新增热缓解潜力列后的新文件（不覆盖原文件）
output_file = os.path.join(
    save_path,
    "情景一_存量+增量_总负荷_附三套热缓解潜力_分年份.xlsx"
)
output_total = os.path.join(
    save_path,
    "情景一_存量+增量_总负荷_附三套热缓解潜力_总表.xlsx"
)

# 【三套余热方案 ERF 配置】
erf_config = {
    "方案1_热缓解潜力(TJ)": {"推理": 0.10, "训练": 0.40},
    "方案2_热缓解潜力(TJ)": {"推理": 0.30, "训练": 0.65},
    "方案3_热缓解潜力(TJ)": {"推理": 0.50, "训练": 0.90}
}

# 【城市分组配置】
# 推理回收潜力组（L1中转城）
infer_cities = ["Chengdu", "Chongqing", "Guiyang", "Shaoguan", "Wuhu", "Zhangjiakou"]
# 训练回收潜力组（L2承接城，优先级更高）
train_cities = ["Guiyang", "Hohhot", "Qingyang", "Zhongwei"]

# 城市名称标准化函数（避免空格、大小写匹配失败）
def standardize_city_name(s):
    if pd.isna(s):
        return s
    return str(s).strip().title()  # 统一首字母大写，消除空格

# ===================== 2. 读取Excel文件 & 处理每个年份Sheet =====================
# 加载工作簿
wb = load_workbook(input_file)
all_sheets = wb.sheetnames
print(f"文件包含的年份Sheet：{all_sheets}")

# 存储所有年份数据，用于生成总表
total_data_list = []

# 遍历每个年份Sheet
for sheet_name in all_sheets:
    ws = wb[sheet_name]
    print(f"\n===== 正在处理：{sheet_name} =====")
    
    # 第一步：读取Sheet数据，识别表头和核心列
    # 读取表头行（第1行）
    header = [cell.value for cell in ws[1]]
    # 识别核心列的列号
    try:
        city_col = header.index("城市") + 1  # Excel列号从1开始
        heat_col = header.index("Stock_Heat (TJ)") + 1
    except ValueError as e:
        print(f"错误：Sheet {sheet_name} 缺失核心列，跳过处理。错误信息：{e}")
        continue
    
    # 第二步：读取所有数据行，标准化城市名称，匹配ERF类型
    max_row = ws.max_row
    city_erf_type = []  # 存储每行对应的ERF类型（推理/训练）
    
    for row in range(2, max_row + 1):
        city_raw = ws.cell(row=row, column=city_col).value
        city_std = standardize_city_name(city_raw)
        
        # 优先级：先匹配训练组，再匹配推理组
        if city_std in train_cities:
            city_erf_type.append("训练")
        elif city_std in infer_cities:
            city_erf_type.append("推理")
        else:
            city_erf_type.append(None)  # 非目标城市，无匹配
    
    # 第三步：计算三套方案的热缓解潜力，写入Sheet
    # 确定新增列的起始列（原表最后一列的下一列）
    start_col = ws.max_column + 1
    
    # 遍历三套方案，逐列写入
    for col_idx, (col_name, erf_dict) in enumerate(erf_config.items()):
        # 写入列名（表头行）
        ws.cell(row=1, column=start_col + col_idx, value=col_name)
        
        # 逐行计算并写入数值
        for row in range(2, max_row + 1):
            erf_type = city_erf_type[row - 2]
            stock_heat = ws.cell(row=row, column=heat_col).value
            
            # 无匹配类型/无热量值，填充0
            if erf_type is None or pd.isna(stock_heat):
                ws.cell(row=row, column=start_col + col_idx, value=0.0)
                continue
            
            # 匹配对应ERF，计算热缓解潜力
            erf_value = erf_dict[erf_type]
            reduce_heat = round(stock_heat * erf_value, 7)  # 保留7位小数，和原表精度一致
            ws.cell(row=row, column=start_col + col_idx, value=reduce_heat)
    
    # 第四步：设置新增列的列宽
    for col in range(start_col, start_col + len(erf_config)):
        col_letter = get_column_letter(col)
        ws.column_dimensions[col_letter].width = 32
    
    # 第五步：读取当前Sheet数据，加入总表列表
    df_year = pd.DataFrame(ws.values)
    df_year.columns = df_year.iloc[0]
    df_year = df_year[1:].reset_index(drop=True)
    df_year["年份"] = sheet_name.replace("年", "")  # 补充年份列
    total_data_list.append(df_year)

# ===================== 3. 保存修改后的分年份Excel =====================
wb.save(output_file)
print(f"\n✅ 分年份Excel处理完成，已保存至：{output_file}")

# ===================== 4. 生成并保存全周期总表 =====================
df_total = pd.concat(total_data_list, ignore_index=True)
# 整理列顺序，把年份列放在最前面
cols_order = ["年份"] + [col for col in df_total.columns if col != "年份"]
df_total = df_total[cols_order]
# 保存总表
df_total.to_excel(output_total, index=False, engine="openpyxl")
print(f"✅ 全周期总表生成完成，已保存至：{output_total}")

# ===================== 5. 结果预览 =====================
print("\n===== 2026年数据预览（核心列） =====")
df_2026 = df_total[df_total["年份"] == "2026"]
preview_cols = [
    "年份", "城市", "Stock_Heat (TJ)",
    "方案1_热缓解潜力(TJ)", "方案2_热缓解潜力(TJ)", "方案3_热缓解潜力(TJ)"
]
print(df_2026[preview_cols].to_string(index=False))

文件包含的年份Sheet：['2026年', '2027年', '2028年', '2029年', '2030年', '2031年', '2032年', '2033年', '2034年', '2035年']

===== 正在处理：2026年 =====

===== 正在处理：2027年 =====

===== 正在处理：2028年 =====

===== 正在处理：2029年 =====

===== 正在处理：2030年 =====

===== 正在处理：2031年 =====

===== 正在处理：2032年 =====

===== 正在处理：2033年 =====

===== 正在处理：2034年 =====

===== 正在处理：2035年 =====

✅ 分年份Excel处理完成，已保存至：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景一_存量+增量_总负荷_附三套热缓解潜力_分年份.xlsx
✅ 全周期总表生成完成，已保存至：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景一_存量+增量_总负荷_附三套热缓解潜力_总表.xlsx

===== 2026年数据预览（核心列） =====
  年份          城市 Stock_Heat (TJ) 方案1_热缓解潜力(TJ) 方案2_热缓解潜力(TJ) 方案3_热缓解潜力(TJ)
2026     Chengdu      265.886839     26.588684     79.766052     132.94342
2026   Chongqing      213.267015     21.326701     63.980104    106.633507
2026     Guiyang     42491.24879  16996.499516  27619.311713  38242.123911
2026      Hohhot     51963.04919  20785.219676  33775.981973  46766.744271
2026    Qingyang    67985.999546  27194.399818  44190.899705  61187.399592
2026    S

In [52]:
import pandas as pd
import numpy as np
import os
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter

# ===================== 1. 基础配置 & 核心参数 =====================
# 统一文件路径
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
os.makedirs(save_path, exist_ok=True)

# 输入文件：情景一 存量+增量 分年份Excel
input_file = os.path.join(
    save_path,
    "情景二_存量+增量_总负荷_分年份.xlsx"
)

# 输出文件：新增热缓解潜力列后的新文件（不覆盖原文件）
output_file = os.path.join(
    save_path,
    "情景二_存量+增量_总负荷_附三套热缓解潜力_分年份.xlsx"
)
output_total = os.path.join(
    save_path,
    "情景二_存量+增量_总负荷_附三套热缓解潜力_总表.xlsx"
)

# 【三套余热方案 ERF 配置】
erf_config = {
    "方案1_热缓解潜力(TJ)": {"推理": 0.10, "训练": 0.40},
    "方案2_热缓解潜力(TJ)": {"推理": 0.30, "训练": 0.65},
    "方案3_热缓解潜力(TJ)": {"推理": 0.50, "训练": 0.90}
}

# 【城市分组配置】
# 推理回收潜力组（L1中转城）
infer_cities = ["Chengdu", "Chongqing", "Guiyang", "Shaoguan", "Wuhu", "Zhangjiakou"]
# 训练回收潜力组（L2承接城，优先级更高）
train_cities = ["Guiyang", "Hohhot", "Qingyang", "Zhongwei"]

# 城市名称标准化函数（避免空格、大小写匹配失败）
def standardize_city_name(s):
    if pd.isna(s):
        return s
    return str(s).strip().title()  # 统一首字母大写，消除空格

# ===================== 2. 读取Excel文件 & 处理每个年份Sheet =====================
# 加载工作簿
wb = load_workbook(input_file)
all_sheets = wb.sheetnames
print(f"文件包含的年份Sheet：{all_sheets}")

# 存储所有年份数据，用于生成总表
total_data_list = []

# 遍历每个年份Sheet
for sheet_name in all_sheets:
    ws = wb[sheet_name]
    print(f"\n===== 正在处理：{sheet_name} =====")
    
    # 第一步：读取Sheet数据，识别表头和核心列
    # 读取表头行（第1行）
    header = [cell.value for cell in ws[1]]
    # 识别核心列的列号
    try:
        city_col = header.index("城市") + 1  # Excel列号从1开始
        heat_col = header.index("Stock_Heat (TJ)") + 1
    except ValueError as e:
        print(f"错误：Sheet {sheet_name} 缺失核心列，跳过处理。错误信息：{e}")
        continue
    
    # 第二步：读取所有数据行，标准化城市名称，匹配ERF类型
    max_row = ws.max_row
    city_erf_type = []  # 存储每行对应的ERF类型（推理/训练）
    
    for row in range(2, max_row + 1):
        city_raw = ws.cell(row=row, column=city_col).value
        city_std = standardize_city_name(city_raw)
        
        # 优先级：先匹配训练组，再匹配推理组
        if city_std in train_cities:
            city_erf_type.append("训练")
        elif city_std in infer_cities:
            city_erf_type.append("推理")
        else:
            city_erf_type.append(None)  # 非目标城市，无匹配
    
    # 第三步：计算三套方案的热缓解潜力，写入Sheet
    # 确定新增列的起始列（原表最后一列的下一列）
    start_col = ws.max_column + 1
    
    # 遍历三套方案，逐列写入
    for col_idx, (col_name, erf_dict) in enumerate(erf_config.items()):
        # 写入列名（表头行）
        ws.cell(row=1, column=start_col + col_idx, value=col_name)
        
        # 逐行计算并写入数值
        for row in range(2, max_row + 1):
            erf_type = city_erf_type[row - 2]
            stock_heat = ws.cell(row=row, column=heat_col).value
            
            # 无匹配类型/无热量值，填充0
            if erf_type is None or pd.isna(stock_heat):
                ws.cell(row=row, column=start_col + col_idx, value=0.0)
                continue
            
            # 匹配对应ERF，计算热缓解潜力
            erf_value = erf_dict[erf_type]
            reduce_heat = round(stock_heat * erf_value, 7)  # 保留7位小数，和原表精度一致
            ws.cell(row=row, column=start_col + col_idx, value=reduce_heat)
    
    # 第四步：设置新增列的列宽
    for col in range(start_col, start_col + len(erf_config)):
        col_letter = get_column_letter(col)
        ws.column_dimensions[col_letter].width = 32
    
    # 第五步：读取当前Sheet数据，加入总表列表
    df_year = pd.DataFrame(ws.values)
    df_year.columns = df_year.iloc[0]
    df_year = df_year[1:].reset_index(drop=True)
    df_year["年份"] = sheet_name.replace("年", "")  # 补充年份列
    total_data_list.append(df_year)

# ===================== 3. 保存修改后的分年份Excel =====================
wb.save(output_file)
print(f"\n✅ 分年份Excel处理完成，已保存至：{output_file}")

# ===================== 4. 生成并保存全周期总表 =====================
df_total = pd.concat(total_data_list, ignore_index=True)
# 整理列顺序，把年份列放在最前面
cols_order = ["年份"] + [col for col in df_total.columns if col != "年份"]
df_total = df_total[cols_order]
# 保存总表
df_total.to_excel(output_total, index=False, engine="openpyxl")
print(f"✅ 全周期总表生成完成，已保存至：{output_total}")

# ===================== 5. 结果预览 =====================
print("\n===== 2026年数据预览（核心列） =====")
df_2026 = df_total[df_total["年份"] == "2026"]
preview_cols = [
    "年份", "城市", "Stock_Heat (TJ)",
    "方案1_热缓解潜力(TJ)", "方案2_热缓解潜力(TJ)", "方案3_热缓解潜力(TJ)"
]
print(df_2026[preview_cols].to_string(index=False))

文件包含的年份Sheet：['2026年', '2027年', '2028年', '2029年', '2030年', '2031年', '2032年', '2033年', '2034年', '2035年']

===== 正在处理：2026年 =====

===== 正在处理：2027年 =====

===== 正在处理：2028年 =====

===== 正在处理：2029年 =====

===== 正在处理：2030年 =====

===== 正在处理：2031年 =====

===== 正在处理：2032年 =====

===== 正在处理：2033年 =====

===== 正在处理：2034年 =====

===== 正在处理：2035年 =====

✅ 分年份Excel处理完成，已保存至：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景二_存量+增量_总负荷_附三套热缓解潜力_分年份.xlsx
✅ 全周期总表生成完成，已保存至：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景二_存量+增量_总负荷_附三套热缓解潜力_总表.xlsx

===== 2026年数据预览（核心列） =====
  年份          城市 Stock_Heat (TJ) 方案1_热缓解潜力(TJ) 方案2_热缓解潜力(TJ) 方案3_热缓解潜力(TJ)
2026     Chengdu      265.886839     26.588684     79.766052     132.94342
2026   Chongqing      213.267015     21.326701     63.980104    106.633507
2026     Guiyang     42491.24879  16996.499516  27619.311713  38242.123911
2026      Hohhot     51963.04919  20785.219676  33775.981973  46766.744271
2026    Qingyang    67985.999546  27194.399818  44190.899705  61187.399592
2026    S

In [54]:
import pandas as pd
import os

save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
increment_file = os.path.join(save_path, "情景一_增量转移_附三套余热可减少热量_总表.xlsx")
output_file = os.path.join(save_path, "情景一_增量转移_附余热可减少+剩余热量_总表.xlsx")

# 读取增量表
df = pd.read_excel(increment_file)

# 补充剩余排放热量列
df["方案1_剩余排放热量(TJ)"] = (df["Heat (TJ)"] - df["方案1_可减少热量(TJ)"]).round(7)
df["方案2_剩余排放热量(TJ)"] = (df["Heat (TJ)"] - df["方案2_可减少热量(TJ)"]).round(7)
df["方案3_剩余排放热量(TJ)"] = (df["Heat (TJ)"] - df["方案3_可减少热量(TJ)"]).round(7)

# 保存文件
df.to_excel(output_file, index=False, engine="openpyxl")
print(f"✅ 增量表已补充剩余排放热量列，保存至：{output_file}")

✅ 增量表已补充剩余排放热量列，保存至：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景一_增量转移_附余热可减少+剩余热量_总表.xlsx


In [60]:
import pandas as pd
import os

save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
increment_file = os.path.join(save_path, "情景一_存量+增量_总负荷_附三套热缓解潜力_总表.xlsx")
output_file = os.path.join(save_path, "情景一_存量转移_附余热可减少+剩余热量_总表.xlsx")

# 读取增量表
df = pd.read_excel(increment_file)

# 补充剩余排放热量列
df["方案1_剩余排放热量(TJ)"] = (df["Stock_Heat (TJ)"] - df["方案1_热缓解潜力(TJ)"]).round(7)
df["方案2_剩余排放热量(TJ)"] = (df["Stock_Heat (TJ)"] - df["方案2_热缓解潜力(TJ)"]).round(7)
df["方案3_剩余排放热量(TJ)"] = (df["Stock_Heat (TJ)"] - df["方案3_热缓解潜力(TJ)"]).round(7)

# 保存文件
df.to_excel(output_file, index=False, engine="openpyxl")
print(f"✅ 存量表已补充剩余排放热量列，保存至：{output_file}")

✅ 存量表已补充剩余排放热量列，保存至：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景一_存量转移_附余热可减少+剩余热量_总表.xlsx


In [55]:
import pandas as pd
import os

save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
increment_file = os.path.join(save_path, "情景二_增量转移_附三套余热可减少热量_总表.xlsx")
output_file = os.path.join(save_path, "情景二_增量转移_附余热可减少+剩余热量_总表.xlsx")

# 读取增量表
df = pd.read_excel(increment_file)

# 补充剩余排放热量列
df["方案1_剩余排放热量(TJ)"] = (df["Heat (TJ)"] - df["方案1_可减少热量(TJ)"]).round(7)
df["方案2_剩余排放热量(TJ)"] = (df["Heat (TJ)"] - df["方案2_可减少热量(TJ)"]).round(7)
df["方案3_剩余排放热量(TJ)"] = (df["Heat (TJ)"] - df["方案3_可减少热量(TJ)"]).round(7)

# 保存文件
df.to_excel(output_file, index=False, engine="openpyxl")
print(f"✅ 增量表已补充剩余排放热量列，保存至：{output_file}")

✅ 增量表已补充剩余排放热量列，保存至：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景二_增量转移_附余热可减少+剩余热量_总表.xlsx


In [74]:
import pandas as pd
import os
from openpyxl import load_workbook

# 你指定的文件路径
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
increment_file = os.path.join(save_path, "情景一_增量转移_附余热可减少+剩余热量_总表.xlsx")
stock_file = os.path.join(save_path, "情景一_存量转移_附余热可减少+剩余热量_总表.xlsx")

# 诊断函数：打印Excel的Sheet、列名、数据样例
def diagnose_excel(file_path, file_name):
    print(f"\n===================== 诊断文件：{file_name} =====================")
    # 检查文件是否存在
    if not os.path.exists(file_path):
        print(f"❌ 错误：文件不存在！路径：{file_path}")
        return
    print(f"✅ 文件存在，路径：{file_path}")
    
    # 读取所有Sheet名
    wb = load_workbook(file_path, read_only=True)
    sheet_names = wb.sheetnames
    print(f"\n📋 所有Sheet名称：{sheet_names}")
    wb.close()
    
    # 遍历每个Sheet，打印列名和前5行数据
    for sheet in sheet_names:
        print(f"\n------ Sheet：{sheet} ------")
        try:
            df = pd.read_excel(file_path, sheet_name=sheet, nrows=5)
            print(f"列名：{df.columns.tolist()}")
            print("前5行数据：")
            print(df.head())
        except Exception as e:
            print(f"❌ 读取Sheet失败：{str(e)}")

# 执行诊断
diagnose_excel(increment_file, "增量转移Excel")
diagnose_excel(stock_file, "存量转移Excel")


===================== 诊断文件：增量转移Excel =====================
✅ 文件存在，路径：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景一_增量转移_附余热可减少+剩余热量_总表.xlsx

📋 所有Sheet名称：['Sheet1']

------ Sheet：Sheet1 ------
列名：['年份', 'Source', 'Target', 'Type', 'Computing power (PFLOPS)', 'Energy (TWh)', 'Heat (TJ)', 'emission factor (kgCO2/kWh)', 'Carbon emission (Mt)', '方案1_可减少热量(TJ)', '方案2_可减少热量(TJ)', '方案3_可减少热量(TJ)', '方案1_剩余排放热量(TJ)', '方案2_剩余排放热量(TJ)', '方案3_剩余排放热量(TJ)']
前5行数据：
     年份   Source       Target Type  Computing power (PFLOPS)  Energy (TWh)  \
0  2026  Beijing       Hohhot   训练                    2389.4      0.301783   
1  2026  Beijing  Zhangjiakou   推理                    1743.6      0.026051   
2  2026  Beijing     Zhongwei   训练                    1593.0      0.272929   
3  2026  Chengdu      Guiyang   训练                    1991.2      0.260238   
4  2026  Chengdu     Qingyang   训练                    1991.2      0.346984   

     Heat (TJ)  emission factor (kgCO2/kWh)  Carbon emission (Mt)  \
0   944.713127 

In [75]:
import pandas as pd
import numpy as np
import os
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

# ===================== 1. 基础配置 & 你指定的本地文件路径 =====================
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
os.makedirs(save_path, exist_ok=True)

# 输入文件（你的两个Excel，均为单Sheet1结构）
increment_file = os.path.join(save_path, "情景一_增量转移_附余热可减少+剩余热量_总表.xlsx")
stock_file = os.path.join(save_path, "情景一_存量转移_附余热可减少+剩余热量_总表.xlsx")

# 输出文件
out_year_detail = os.path.join(save_path, "情景一_年份+城市_余热总减少+总剩余_明细总表.xlsx")
out_city_total = os.path.join(save_path, "情景一_同城市合并_余热总减少+总剩余_汇总表.xlsx")

# 方案列名配置（和你文件内的列名完全匹配）
scheme_config = {
    "方案1": {
        "inc_reduce": "方案1_可减少热量(TJ)",
        "inc_remain": "方案1_剩余排放热量(TJ)",
        "stock_reduce": "方案1_热缓解潜力(TJ)",
        "stock_remain": "方案1_剩余排放热量(TJ)"
    },
    "方案2": {
        "inc_reduce": "方案2_可减少热量(TJ)",
        "inc_remain": "方案2_剩余排放热量(TJ)",
        "stock_reduce": "方案2_热缓解潜力(TJ)",
        "stock_remain": "方案2_剩余排放热量(TJ)"
    },
    "方案3": {
        "inc_reduce": "方案3_可减少热量(TJ)",
        "inc_remain": "方案3_剩余排放热量(TJ)",
        "stock_reduce": "方案3_热缓解潜力(TJ)",
        "stock_remain": "方案3_剩余排放热量(TJ)"
    }
}

# 城市名称标准化（消除空格、大小写差异，保证匹配成功）
def standardize_city(s):
    if pd.isna(s):
        return s
    return str(s).strip().title()

# ===================== 2. 读取两个Excel的全量数据 =====================
print("===== 开始读取Excel文件 =====")

# 2.1 读取增量转移数据（单Sheet1，城市列：Target）
df_inc = pd.read_excel(increment_file, sheet_name="Sheet1")
# 标准化城市列，补充标准化城市字段
df_inc["标准化城市"] = df_inc["Target"].apply(standardize_city)
print(f"✅ 增量数据读取完成，共{len(df_inc)}行数据")
print(f"包含年份：{sorted(df_inc['年份'].unique())}")
print(f"包含城市：{sorted(df_inc['标准化城市'].unique())}")

# 2.2 读取存量转移数据（单Sheet1，城市列：城市）
df_stock = pd.read_excel(stock_file, sheet_name="Sheet1")
# 标准化城市列，补充标准化城市字段
df_stock["标准化城市"] = df_stock["城市"].apply(standardize_city)
print(f"\n✅ 存量数据读取完成，共{len(df_stock)}行数据")
print(f"包含年份：{sorted(df_stock['年份'].unique())}")
print(f"包含城市：{sorted(df_stock['标准化城市'].unique())}")

# ===================== 3. 按「年份 + 标准化城市」双维度汇总 =====================
group_cols = ["年份", "标准化城市"]
print(f"\n===== 按{group_cols}双维度汇总数据 =====")

# 3.1 增量数据汇总
df_inc_agg = df_inc.groupby(group_cols).agg({
    scheme_config["方案1"]["inc_reduce"]: "sum",
    scheme_config["方案1"]["inc_remain"]: "sum",
    scheme_config["方案2"]["inc_reduce"]: "sum",
    scheme_config["方案2"]["inc_remain"]: "sum",
    scheme_config["方案3"]["inc_reduce"]: "sum",
    scheme_config["方案3"]["inc_remain"]: "sum"
}).reset_index()

# 重命名增量列
df_inc_agg.rename(columns={
    scheme_config["方案1"]["inc_reduce"]: "方案1_增量减少热量(TJ)",
    scheme_config["方案1"]["inc_remain"]: "方案1_增量剩余热量(TJ)",
    scheme_config["方案2"]["inc_reduce"]: "方案2_增量减少热量(TJ)",
    scheme_config["方案2"]["inc_remain"]: "方案2_增量剩余热量(TJ)",
    scheme_config["方案3"]["inc_reduce"]: "方案3_增量减少热量(TJ)",
    scheme_config["方案3"]["inc_remain"]: "方案3_增量剩余热量(TJ)"
}, inplace=True)
print(f"增量数据汇总完成，共{len(df_inc_agg)}行数据")

# 3.2 存量数据汇总
df_stock_agg = df_stock.groupby(group_cols).agg({
    scheme_config["方案1"]["stock_reduce"]: "sum",
    scheme_config["方案1"]["stock_remain"]: "sum",
    scheme_config["方案2"]["stock_reduce"]: "sum",
    scheme_config["方案2"]["stock_remain"]: "sum",
    scheme_config["方案3"]["stock_reduce"]: "sum",
    scheme_config["方案3"]["stock_remain"]: "sum"
}).reset_index()

# 重命名存量列
df_stock_agg.rename(columns={
    scheme_config["方案1"]["stock_reduce"]: "方案1_存量减少热量(TJ)",
    scheme_config["方案1"]["stock_remain"]: "方案1_存量剩余热量(TJ)",
    scheme_config["方案2"]["stock_reduce"]: "方案2_存量减少热量(TJ)",
    scheme_config["方案2"]["stock_remain"]: "方案2_存量剩余热量(TJ)",
    scheme_config["方案3"]["stock_reduce"]: "方案3_存量减少热量(TJ)",
    scheme_config["方案3"]["stock_remain"]: "方案3_存量剩余热量(TJ)"
}, inplace=True)
print(f"存量数据汇总完成，共{len(df_stock_agg)}行数据")

# ===================== 4. 合并增量+存量，同城市同方案合并计算 =====================
# 按年份+标准化城市精准匹配
df_merge = pd.merge(
    df_inc_agg,
    df_stock_agg,
    on=group_cols,
    how="outer"
).fillna(0)

# 统一城市列名
df_merge.rename(columns={"标准化城市": "城市名称"}, inplace=True)

# 计算每个方案的总减少热量、总剩余热量（核心合并逻辑）
for scheme in ["方案1", "方案2", "方案3"]:
    # 总减少热量 = 增量可减少 + 存量热缓解
    df_merge[f"{scheme}_总减少热量(TJ)"] = df_merge[f"{scheme}_增量减少热量(TJ)"] + df_merge[f"{scheme}_存量减少热量(TJ)"]
    # 总剩余热量 = 增量剩余 + 存量剩余
    df_merge[f"{scheme}_总剩余热量(TJ)"] = df_merge[f"{scheme}_增量剩余热量(TJ)"] + df_merge[f"{scheme}_存量剩余热量(TJ)"]

# 整理最终明细列结构（年份+城市在前，核心指标在后）
detail_cols = [
    "年份", "城市名称",
    "方案1_总减少热量(TJ)", "方案2_总减少热量(TJ)", "方案3_总减少热量(TJ)",
    "方案1_总剩余热量(TJ)", "方案2_总剩余热量(TJ)", "方案3_总剩余热量(TJ)"
]
df_year_detail = df_merge[detail_cols].copy()

# 数值精度统一：保留7位小数
for col in detail_cols[2:]:
    df_year_detail[col] = df_year_detail[col].round(7)

# 按年份+城市排序
df_year_detail = df_year_detail.sort_values(by=["年份", "城市名称"]).reset_index(drop=True)

# ===================== 5. 生成全年份城市总汇总表（全局统计） =====================
df_city_total = df_year_detail.groupby("城市名称").agg({
    "方案1_总减少热量(TJ)": "sum",
    "方案2_总减少热量(TJ)": "sum",
    "方案3_总减少热量(TJ)": "sum",
    "方案1_总剩余热量(TJ)": "sum",
    "方案2_总剩余热量(TJ)": "sum",
    "方案3_总剩余热量(TJ)": "sum"
}).reset_index()

# 数值精度统一
for col in df_city_total.columns[1:]:
    df_city_total[col] = df_city_total[col].round(7)

# 按城市排序
df_city_total = df_city_total.sort_values(by="城市名称").reset_index(drop=True)

# ===================== 6. 保存所有文件到你指定的目录 =====================
# 6.1 保存 年份+城市 明细总表（带年份维度，用于趋势分析）
df_year_detail.to_excel(out_year_detail, index=False, engine="openpyxl")
print(f"\n✅  年份+城市明细总表已保存至：{out_year_detail}")

# 6.2 保存 全年份城市总汇总表（论文核心附表）
wb_city = Workbook()
wb_city.remove(wb_city.active)
ws_city = wb_city.create_sheet(title="城市总汇总表")
for row in dataframe_to_rows(df_city_total, index=False, header=True):
    ws_city.append(row)
# 统一列宽
for col_idx in range(1, ws_city.max_column + 1):
    col_letter = ws_city.cell(row=1, column=col_idx).column_letter
    ws_city.column_dimensions[col_letter].width = 32
wb_city.save(out_city_total)
print(f"✅  全年份城市总汇总表已保存至：{out_city_total}")

# ===================== 7. 结果预览与校验 =====================
print("\n===== 数据完整性校验 =====")
print(f"包含年份：{sorted(df_year_detail['年份'].unique())}")
print(f"包含城市：{sorted(df_year_detail['城市名称'].unique())}")
print(f"总数据行数：{len(df_year_detail)}")
print(f"每年数据行数：{df_year_detail.groupby('年份').size().to_dict()}")

print("\n===== 2026年 核心数据预览 =====")
print(df_year_detail[df_year_detail["年份"] == 2026].to_string(index=False))

print("\n===== 全年份城市总汇总预览 =====")
print(df_city_total.to_string(index=False))

===== 开始读取Excel文件 =====
✅ 增量数据读取完成，共210行数据
包含年份：[np.int64(2026), np.int64(2027), np.int64(2028), np.int64(2029), np.int64(2030), np.int64(2031), np.int64(2032), np.int64(2033), np.int64(2034), np.int64(2035)]
包含城市：['Chengdu', 'Chongqing', 'Guiyang', 'Hohhot', 'Qingyang', 'Shaoguan', 'Wuhu', 'Zhangjiakou', 'Zhongwei']

✅ 存量数据读取完成，共90行数据
包含年份：[np.int64(2026), np.int64(2027), np.int64(2028), np.int64(2029), np.int64(2030), np.int64(2031), np.int64(2032), np.int64(2033), np.int64(2034), np.int64(2035)]
包含城市：['Chengdu', 'Chongqing', 'Guiyang', 'Hohhot', 'Qingyang', 'Shaoguan', 'Wuhu', 'Zhangjiakou', 'Zhongwei']

===== 按['年份', '标准化城市']双维度汇总数据 =====
增量数据汇总完成，共90行数据
存量数据汇总完成，共90行数据

✅  年份+城市明细总表已保存至：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景一_年份+城市_余热总减少+总剩余_明细总表.xlsx
✅  全年份城市总汇总表已保存至：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景一_同城市合并_余热总减少+总剩余_汇总表.xlsx

===== 数据完整性校验 =====
包含年份：[np.int64(2026), np.int64(2027), np.int64(2028), np.int64(2029), np.int64(2030), np.int64(2031), np.int64(2032), np.int64(2033), 

In [76]:
import pandas as pd
import numpy as np
import os
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

# ===================== 1. 基础配置 & 你指定的本地文件路径 =====================
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
os.makedirs(save_path, exist_ok=True)

# 输入文件（你的两个Excel，均为单Sheet1结构）
increment_file = os.path.join(save_path, "情景二_增量转移_附余热可减少+剩余热量_总表.xlsx")
stock_file = os.path.join(save_path, "情景一_存量转移_附余热可减少+剩余热量_总表.xlsx")

# 输出文件
out_year_detail = os.path.join(save_path, "情景二_年份+城市_余热总减少+总剩余_明细总表.xlsx")
out_city_total = os.path.join(save_path, "情景二_同城市合并_余热总减少+总剩余_汇总表.xlsx")

# 方案列名配置（和你文件内的列名完全匹配）
scheme_config = {
    "方案1": {
        "inc_reduce": "方案1_可减少热量(TJ)",
        "inc_remain": "方案1_剩余排放热量(TJ)",
        "stock_reduce": "方案1_热缓解潜力(TJ)",
        "stock_remain": "方案1_剩余排放热量(TJ)"
    },
    "方案2": {
        "inc_reduce": "方案2_可减少热量(TJ)",
        "inc_remain": "方案2_剩余排放热量(TJ)",
        "stock_reduce": "方案2_热缓解潜力(TJ)",
        "stock_remain": "方案2_剩余排放热量(TJ)"
    },
    "方案3": {
        "inc_reduce": "方案3_可减少热量(TJ)",
        "inc_remain": "方案3_剩余排放热量(TJ)",
        "stock_reduce": "方案3_热缓解潜力(TJ)",
        "stock_remain": "方案3_剩余排放热量(TJ)"
    }
}

# 城市名称标准化（消除空格、大小写差异，保证匹配成功）
def standardize_city(s):
    if pd.isna(s):
        return s
    return str(s).strip().title()

# ===================== 2. 读取两个Excel的全量数据 =====================
print("===== 开始读取Excel文件 =====")

# 2.1 读取增量转移数据（单Sheet1，城市列：Target）
df_inc = pd.read_excel(increment_file, sheet_name="Sheet1")
# 标准化城市列，补充标准化城市字段
df_inc["标准化城市"] = df_inc["Target"].apply(standardize_city)
print(f"✅ 增量数据读取完成，共{len(df_inc)}行数据")
print(f"包含年份：{sorted(df_inc['年份'].unique())}")
print(f"包含城市：{sorted(df_inc['标准化城市'].unique())}")

# 2.2 读取存量转移数据（单Sheet1，城市列：城市）
df_stock = pd.read_excel(stock_file, sheet_name="Sheet1")
# 标准化城市列，补充标准化城市字段
df_stock["标准化城市"] = df_stock["城市"].apply(standardize_city)
print(f"\n✅ 存量数据读取完成，共{len(df_stock)}行数据")
print(f"包含年份：{sorted(df_stock['年份'].unique())}")
print(f"包含城市：{sorted(df_stock['标准化城市'].unique())}")

# ===================== 3. 按「年份 + 标准化城市」双维度汇总 =====================
group_cols = ["年份", "标准化城市"]
print(f"\n===== 按{group_cols}双维度汇总数据 =====")

# 3.1 增量数据汇总
df_inc_agg = df_inc.groupby(group_cols).agg({
    scheme_config["方案1"]["inc_reduce"]: "sum",
    scheme_config["方案1"]["inc_remain"]: "sum",
    scheme_config["方案2"]["inc_reduce"]: "sum",
    scheme_config["方案2"]["inc_remain"]: "sum",
    scheme_config["方案3"]["inc_reduce"]: "sum",
    scheme_config["方案3"]["inc_remain"]: "sum"
}).reset_index()

# 重命名增量列
df_inc_agg.rename(columns={
    scheme_config["方案1"]["inc_reduce"]: "方案1_增量减少热量(TJ)",
    scheme_config["方案1"]["inc_remain"]: "方案1_增量剩余热量(TJ)",
    scheme_config["方案2"]["inc_reduce"]: "方案2_增量减少热量(TJ)",
    scheme_config["方案2"]["inc_remain"]: "方案2_增量剩余热量(TJ)",
    scheme_config["方案3"]["inc_reduce"]: "方案3_增量减少热量(TJ)",
    scheme_config["方案3"]["inc_remain"]: "方案3_增量剩余热量(TJ)"
}, inplace=True)
print(f"增量数据汇总完成，共{len(df_inc_agg)}行数据")

# 3.2 存量数据汇总
df_stock_agg = df_stock.groupby(group_cols).agg({
    scheme_config["方案1"]["stock_reduce"]: "sum",
    scheme_config["方案1"]["stock_remain"]: "sum",
    scheme_config["方案2"]["stock_reduce"]: "sum",
    scheme_config["方案2"]["stock_remain"]: "sum",
    scheme_config["方案3"]["stock_reduce"]: "sum",
    scheme_config["方案3"]["stock_remain"]: "sum"
}).reset_index()

# 重命名存量列
df_stock_agg.rename(columns={
    scheme_config["方案1"]["stock_reduce"]: "方案1_存量减少热量(TJ)",
    scheme_config["方案1"]["stock_remain"]: "方案1_存量剩余热量(TJ)",
    scheme_config["方案2"]["stock_reduce"]: "方案2_存量减少热量(TJ)",
    scheme_config["方案2"]["stock_remain"]: "方案2_存量剩余热量(TJ)",
    scheme_config["方案3"]["stock_reduce"]: "方案3_存量减少热量(TJ)",
    scheme_config["方案3"]["stock_remain"]: "方案3_存量剩余热量(TJ)"
}, inplace=True)
print(f"存量数据汇总完成，共{len(df_stock_agg)}行数据")

# ===================== 4. 合并增量+存量，同城市同方案合并计算 =====================
# 按年份+标准化城市精准匹配
df_merge = pd.merge(
    df_inc_agg,
    df_stock_agg,
    on=group_cols,
    how="outer"
).fillna(0)

# 统一城市列名
df_merge.rename(columns={"标准化城市": "城市名称"}, inplace=True)

# 计算每个方案的总减少热量、总剩余热量（核心合并逻辑）
for scheme in ["方案1", "方案2", "方案3"]:
    # 总减少热量 = 增量可减少 + 存量热缓解
    df_merge[f"{scheme}_总减少热量(TJ)"] = df_merge[f"{scheme}_增量减少热量(TJ)"] + df_merge[f"{scheme}_存量减少热量(TJ)"]
    # 总剩余热量 = 增量剩余 + 存量剩余
    df_merge[f"{scheme}_总剩余热量(TJ)"] = df_merge[f"{scheme}_增量剩余热量(TJ)"] + df_merge[f"{scheme}_存量剩余热量(TJ)"]

# 整理最终明细列结构（年份+城市在前，核心指标在后）
detail_cols = [
    "年份", "城市名称",
    "方案1_总减少热量(TJ)", "方案2_总减少热量(TJ)", "方案3_总减少热量(TJ)",
    "方案1_总剩余热量(TJ)", "方案2_总剩余热量(TJ)", "方案3_总剩余热量(TJ)"
]
df_year_detail = df_merge[detail_cols].copy()

# 数值精度统一：保留7位小数
for col in detail_cols[2:]:
    df_year_detail[col] = df_year_detail[col].round(7)

# 按年份+城市排序
df_year_detail = df_year_detail.sort_values(by=["年份", "城市名称"]).reset_index(drop=True)

# ===================== 5. 生成全年份城市总汇总表（全局统计） =====================
df_city_total = df_year_detail.groupby("城市名称").agg({
    "方案1_总减少热量(TJ)": "sum",
    "方案2_总减少热量(TJ)": "sum",
    "方案3_总减少热量(TJ)": "sum",
    "方案1_总剩余热量(TJ)": "sum",
    "方案2_总剩余热量(TJ)": "sum",
    "方案3_总剩余热量(TJ)": "sum"
}).reset_index()

# 数值精度统一
for col in df_city_total.columns[1:]:
    df_city_total[col] = df_city_total[col].round(7)

# 按城市排序
df_city_total = df_city_total.sort_values(by="城市名称").reset_index(drop=True)

# ===================== 6. 保存所有文件到你指定的目录 =====================
# 6.1 保存 年份+城市 明细总表（带年份维度，用于趋势分析）
df_year_detail.to_excel(out_year_detail, index=False, engine="openpyxl")
print(f"\n✅  年份+城市明细总表已保存至：{out_year_detail}")

# 6.2 保存 全年份城市总汇总表（论文核心附表）
wb_city = Workbook()
wb_city.remove(wb_city.active)
ws_city = wb_city.create_sheet(title="城市总汇总表")
for row in dataframe_to_rows(df_city_total, index=False, header=True):
    ws_city.append(row)
# 统一列宽
for col_idx in range(1, ws_city.max_column + 1):
    col_letter = ws_city.cell(row=1, column=col_idx).column_letter
    ws_city.column_dimensions[col_letter].width = 32
wb_city.save(out_city_total)
print(f"✅  全年份城市总汇总表已保存至：{out_city_total}")

# ===================== 7. 结果预览与校验 =====================
print("\n===== 数据完整性校验 =====")
print(f"包含年份：{sorted(df_year_detail['年份'].unique())}")
print(f"包含城市：{sorted(df_year_detail['城市名称'].unique())}")
print(f"总数据行数：{len(df_year_detail)}")
print(f"每年数据行数：{df_year_detail.groupby('年份').size().to_dict()}")

print("\n===== 2026年 核心数据预览 =====")
print(df_year_detail[df_year_detail["年份"] == 2026].to_string(index=False))

print("\n===== 全年份城市总汇总预览 =====")
print(df_city_total.to_string(index=False))

===== 开始读取Excel文件 =====
✅ 增量数据读取完成，共270行数据
包含年份：[np.int64(2026), np.int64(2027), np.int64(2028), np.int64(2029), np.int64(2030), np.int64(2031), np.int64(2032), np.int64(2033), np.int64(2034), np.int64(2035)]
包含城市：['Chengdu', 'Chongqing', 'Guiyang', 'Hohhot', 'Qingyang', 'Shaoguan', 'Wuhu', 'Zhangjiakou', 'Zhongwei']

✅ 存量数据读取完成，共90行数据
包含年份：[np.int64(2026), np.int64(2027), np.int64(2028), np.int64(2029), np.int64(2030), np.int64(2031), np.int64(2032), np.int64(2033), np.int64(2034), np.int64(2035)]
包含城市：['Chengdu', 'Chongqing', 'Guiyang', 'Hohhot', 'Qingyang', 'Shaoguan', 'Wuhu', 'Zhangjiakou', 'Zhongwei']

===== 按['年份', '标准化城市']双维度汇总数据 =====
增量数据汇总完成，共90行数据
存量数据汇总完成，共90行数据

✅  年份+城市明细总表已保存至：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景二_年份+城市_余热总减少+总剩余_明细总表.xlsx
✅  全年份城市总汇总表已保存至：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景二_同城市合并_余热总减少+总剩余_汇总表.xlsx

===== 数据完整性校验 =====
包含年份：[np.int64(2026), np.int64(2027), np.int64(2028), np.int64(2029), np.int64(2030), np.int64(2031), np.int64(2032), np.int64(2033), 

In [86]:
import pandas as pd
import numpy as np
import os

# ===================== 1. 基础配置 & 固定参数（全文统一，请勿修改） =====================
# 你指定的本地文件目录
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
os.makedirs(save_path, exist_ok=True)

# 输入文件
# 1. 余热明细总表
heat_file = os.path.join(save_path, "情景一_年份+城市_余热总减少+总剩余_明细总表.xlsx")
# 2. 九大城市电网碳排放因子表
grid_file = os.path.join(save_path, "九大城市2026-2035电网碳排放因子_线性插值.csv")

# 输出文件
output_file = os.path.join(save_path, "情景一_年份+城市_三路径碳减排量_明细总表.xlsx")

# 热-碳换算固定参数（学术通用值，和前期研究口径统一）
# 供暖路径参数
Q_std = 29307        # 标准煤低位热值，单位：kJ/kg
eta_boil = 0.85      # 民用燃煤锅炉热效率
EF_coal = 2.6        # 燃煤碳排放因子，单位：tCO2/t标煤
# 制冷路径参数
COP_abs = 0.7         # 单效溴化锂吸收式制冷COP
COP_comp = 3.0        # 常规电驱动压缩式制冷COP
# 供电路径参数
eta_orc = 0.15        # 低品位余热ORC有机朗肯循环发电效率

# 余热方案列表
scheme_list = ["方案1", "方案2", "方案3"]

# 城市名称标准化函数（消除空格、大小写差异，保证匹配成功）
def standardize_city(s):
    if pd.isna(s):
        return s
    return str(s).strip().title()  # 统一首字母大写，消除首尾空格

# ===================== 2. 读取输入文件 & 数据标准化 =====================
print("===== 开始读取输入文件 =====")

# 2.1 读取余热明细总表
df_heat = pd.read_excel(heat_file)
# 标准化城市名称、年份格式
df_heat["城市名称"] = df_heat["城市名称"].apply(standardize_city)
df_heat["年份"] = df_heat["年份"].astype(int)  # 统一为数字年份
print(f"✅ 余热表读取完成，共{len(df_heat)}行数据")
print(f"包含城市：{sorted(df_heat['城市名称'].unique())}")
print(f"包含年份：{sorted(df_heat['年份'].unique())}")

# 2.2 读取电网碳排放因子表
df_grid = pd.read_csv(grid_file)
# 标准化城市名称、年份格式
df_grid["城市"] = df_grid["城市"].apply(standardize_city)
df_grid["年份"] = df_grid["年份"].astype(int)
# 保留核心匹配列
df_grid = df_grid[["城市", "年份", "电网碳排放因子"]].copy()
print(f"\n✅ 电网因子表读取完成，共{len(df_grid)}行数据")
print(f"包含城市：{sorted(df_grid['城市'].unique())}")
print(f"包含年份：{sorted(df_grid['年份'].unique())}")

# ===================== 3. 按「年份 + 城市」匹配合并数据 =====================
# 合并两个表，把电网碳排放因子匹配到对应年份+城市
df_merge = pd.merge(
    df_heat,
    df_grid,
    left_on=["年份", "城市名称"],
    right_on=["年份", "城市"],
    how="left"
)

# 匹配不到的电网因子填充0，避免计算错误
df_merge["电网碳排放因子"] = df_merge["电网碳排放因子"].fillna(0)
# 移除重复的城市列
df_merge = df_merge.drop(columns=["城市"])

print(f"\n✅ 数据匹配完成，合并后共{len(df_merge)}行数据")
print(f"电网因子匹配成功率：{(df_merge['电网碳排放因子'] > 0).sum() / len(df_merge) * 100:.2f}%")

# ===================== 4. 分方案、分路径计算碳减排量 =====================
# 循环每个余热方案，计算3条路径的碳减排量
for scheme in scheme_list:
    # 当前方案的总减少热量（核心输入）
    reduce_heat_col = f"{scheme}_总减少热量(TJ)"
    print(f"\n===== 正在计算{scheme}的三路径碳减排量 =====")

    # ---------------------- 路径1：余热供暖（替代燃煤锅炉） ----------------------
    # 公式：C_供暖(Mt) = (Q_rec × 10^9 × EF_coal) / (Q_std × 1000*η_boil × 10^6)
    heating_col = f"{scheme}_供暖_碳减排量(Mt)"
    df_merge[heating_col] = (
        df_merge[reduce_heat_col] * 10**9 * EF_coal
    ) / (Q_std * eta_boil * 10**9)
    # 保留7位小数，和原表精度统一
    df_merge[heating_col] = df_merge[heating_col].round(7)

    # ---------------------- 路径2：余热制冷（吸收式制冷替代电制冷） ----------------------
    # 步骤1：余热转化冷量 Q_cool = Q_rec × COP_abs
    cool_energy = df_merge[reduce_heat_col] * COP_abs
    # 步骤2：计算节省电量 E_save(kWh) = (Q_cool × 10^9) / (3600 × COP_comp)
    save_power = (cool_energy * 10**9) / (3600 * COP_comp)
    # 步骤3：计算碳减排 C_制冷(Mt) = (E_save × 电网因子) / 10^6
    cooling_col = f"{scheme}_制冷_碳减排量(Mt)"
    df_merge[cooling_col] = (save_power * df_merge["电网碳排放因子"]) / 10**9
    df_merge[cooling_col] = df_merge[cooling_col].round(7)

    # ---------------------- 路径3：余热供电（ORC发电替代电网电） ----------------------
    # 步骤1：计算发电量 E_gen(kWh) = (Q_rec × 10^9 × η_orc) / 3600
    gen_power = (df_merge[reduce_heat_col] * 10**9 * eta_orc) / 3600
    # 步骤2：计算碳减排 C_供电(Mt) = (E_gen × 电网因子) / 10^6
    power_col = f"{scheme}_供电_碳减排量(Mt)"
    df_merge[power_col] = (gen_power * df_merge["电网碳排放因子"]) / 10**9
    df_merge[power_col] = df_merge[power_col].round(7)

# ===================== 5. 整理列顺序 & 保存最终文件 =====================
# 构建最终列顺序：原表列 → 电网因子 → 各方案三路径减排列
base_cols = [
    "年份", "城市名称",
    "方案1_总减少热量(TJ)", "方案2_总减少热量(TJ)", "方案3_总减少热量(TJ)",
    "方案1_总剩余热量(TJ)", "方案2_总剩余热量(TJ)", "方案3_总剩余热量(TJ)",
    "电网碳排放因子"
]

# 追加各方案三路径减排列
add_cols = []
for scheme in scheme_list:
    add_cols.extend([
        f"{scheme}_供暖_碳减排量(Mt)",
        f"{scheme}_制冷_碳减排量(Mt)",
        f"{scheme}_供电_碳减排量(Mt)"
    ])

final_cols = base_cols + add_cols
df_final = df_merge[final_cols].copy()

# 按年份+城市排序
df_final = df_final.sort_values(by=["年份", "城市名称"]).reset_index(drop=True)

# 保存最终Excel文件
df_final.to_excel(output_file, index=False, engine="openpyxl")
print(f"\n✅ 全部计算完成！最终文件已保存至：{output_file}")

# ===================== 6. 结果预览 & 校验 =====================
print("\n===== 2026年 核心结果预览 =====")
preview_cols = [
    "年份", "城市名称", "方案2_总减少热量(TJ)",
    "方案2_供暖_碳减排量(Mt)", "方案2_制冷_碳减排量(Mt)", "方案2_供电_碳减排量(Mt)"
]
print(df_final[df_final["年份"] == 2026][preview_cols].to_string(index=False))

print("\n===== 全量数据统计（方案2 基准情景） =====")
total_heating = df_final["方案2_供暖_碳减排量(Mt)"].sum()
total_cooling = df_final["方案2_制冷_碳减排量(Mt)"].sum()
total_power = df_final["方案2_供电_碳减排量(Mt)"].sum()
print(f"2026-2035年 供暖路径总减排：{total_heating:.4f} Mt")
print(f"2026-2035年 制冷路径总减排：{total_cooling:.4f} Mt")
print(f"2026-2035年 供电路径总减排：{total_power:.4f} Mt")
print(f"同量余热下，供暖减排是制冷的{total_heating/total_cooling:.2f}倍，是供电的{total_heating/total_power:.2f}倍")

===== 开始读取输入文件 =====
✅ 余热表读取完成，共90行数据
包含城市：['Chengdu', 'Chongqing', 'Guiyang', 'Hohhot', 'Qingyang', 'Shaoguan', 'Wuhu', 'Zhangjiakou', 'Zhongwei']
包含年份：[np.int64(2026), np.int64(2027), np.int64(2028), np.int64(2029), np.int64(2030), np.int64(2031), np.int64(2032), np.int64(2033), np.int64(2034), np.int64(2035)]

✅ 电网因子表读取完成，共90行数据
包含城市：['Chengdu', 'Chongqing', 'Guiyang', 'Hohhot', 'Qingyang', 'Shaoguan', 'Wuhu', 'Zhangjiakou', 'Zhongwei']
包含年份：[np.int64(2026), np.int64(2027), np.int64(2028), np.int64(2029), np.int64(2030), np.int64(2031), np.int64(2032), np.int64(2033), np.int64(2034), np.int64(2035)]

✅ 数据匹配完成，合并后共90行数据
电网因子匹配成功率：100.00%

===== 正在计算方案1的三路径碳减排量 =====

===== 正在计算方案2的三路径碳减排量 =====

===== 正在计算方案3的三路径碳减排量 =====

✅ 全部计算完成！最终文件已保存至：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景一_年份+城市_三路径碳减排量_明细总表.xlsx

===== 2026年 核心结果预览 =====
  年份        城市名称  方案2_总减少热量(TJ)  方案2_供暖_碳减排量(Mt)  方案2_制冷_碳减排量(Mt)  方案2_供电_碳减排量(Mt)
2026     Chengdu      97.993967         0.010228         0.000616         0

In [87]:
import pandas as pd
import numpy as np
import os

# ===================== 1. 基础配置 & 固定参数（全文统一，请勿修改） =====================
# 你指定的本地文件目录
save_path = r"D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理"
os.makedirs(save_path, exist_ok=True)

# 输入文件
# 1. 余热明细总表
heat_file = os.path.join(save_path, "情景二_年份+城市_余热总减少+总剩余_明细总表.xlsx")
# 2. 九大城市电网碳排放因子表
grid_file = os.path.join(save_path, "九大城市2026-2035电网碳排放因子_线性插值.csv")

# 输出文件
output_file = os.path.join(save_path, "情景二_年份+城市_三路径碳减排量_明细总表.xlsx")

# 热-碳换算固定参数（学术通用值，和前期研究口径统一）
# 供暖路径参数
Q_std = 29307        # 标准煤低位热值，单位：kJ/kg
eta_boil = 0.85      # 民用燃煤锅炉热效率
EF_coal = 2.6        # 燃煤碳排放因子，单位：tCO2/t标煤
# 制冷路径参数
COP_abs = 0.7         # 单效溴化锂吸收式制冷COP
COP_comp = 3.0        # 常规电驱动压缩式制冷COP
# 供电路径参数
eta_orc = 0.15        # 低品位余热ORC有机朗肯循环发电效率

# 余热方案列表
scheme_list = ["方案1", "方案2", "方案3"]

# 城市名称标准化函数（消除空格、大小写差异，保证匹配成功）
def standardize_city(s):
    if pd.isna(s):
        return s
    return str(s).strip().title()  # 统一首字母大写，消除首尾空格

# ===================== 2. 读取输入文件 & 数据标准化 =====================
print("===== 开始读取输入文件 =====")

# 2.1 读取余热明细总表
df_heat = pd.read_excel(heat_file)
# 标准化城市名称、年份格式
df_heat["城市名称"] = df_heat["城市名称"].apply(standardize_city)
df_heat["年份"] = df_heat["年份"].astype(int)  # 统一为数字年份
print(f"✅ 余热表读取完成，共{len(df_heat)}行数据")
print(f"包含城市：{sorted(df_heat['城市名称'].unique())}")
print(f"包含年份：{sorted(df_heat['年份'].unique())}")

# 2.2 读取电网碳排放因子表
df_grid = pd.read_csv(grid_file)
# 标准化城市名称、年份格式
df_grid["城市"] = df_grid["城市"].apply(standardize_city)
df_grid["年份"] = df_grid["年份"].astype(int)
# 保留核心匹配列
df_grid = df_grid[["城市", "年份", "电网碳排放因子"]].copy()
print(f"\n✅ 电网因子表读取完成，共{len(df_grid)}行数据")
print(f"包含城市：{sorted(df_grid['城市'].unique())}")
print(f"包含年份：{sorted(df_grid['年份'].unique())}")

# ===================== 3. 按「年份 + 城市」匹配合并数据 =====================
# 合并两个表，把电网碳排放因子匹配到对应年份+城市
df_merge = pd.merge(
    df_heat,
    df_grid,
    left_on=["年份", "城市名称"],
    right_on=["年份", "城市"],
    how="left"
)

# 匹配不到的电网因子填充0，避免计算错误
df_merge["电网碳排放因子"] = df_merge["电网碳排放因子"].fillna(0)
# 移除重复的城市列
df_merge = df_merge.drop(columns=["城市"])

print(f"\n✅ 数据匹配完成，合并后共{len(df_merge)}行数据")
print(f"电网因子匹配成功率：{(df_merge['电网碳排放因子'] > 0).sum() / len(df_merge) * 100:.2f}%")

# ===================== 4. 分方案、分路径计算碳减排量 =====================
# 循环每个余热方案，计算3条路径的碳减排量
for scheme in scheme_list:
    # 当前方案的总减少热量（核心输入）
    reduce_heat_col = f"{scheme}_总减少热量(TJ)"
    print(f"\n===== 正在计算{scheme}的三路径碳减排量 =====")

    # ---------------------- 路径1：余热供暖（替代燃煤锅炉） ----------------------
    # 公式：C_供暖(Mt) = (Q_rec × 10^9 × EF_coal) / (Q_std × 1000*η_boil × 10^6)
    heating_col = f"{scheme}_供暖_碳减排量(Mt)"
    df_merge[heating_col] = (
        df_merge[reduce_heat_col] * 10**9 * EF_coal
    ) / (Q_std * eta_boil * 10**9)
    # 保留7位小数，和原表精度统一
    df_merge[heating_col] = df_merge[heating_col].round(7)

    # ---------------------- 路径2：余热制冷（吸收式制冷替代电制冷） ----------------------
    # 步骤1：余热转化冷量 Q_cool = Q_rec × COP_abs
    cool_energy = df_merge[reduce_heat_col] * COP_abs
    # 步骤2：计算节省电量 E_save(kWh) = (Q_cool × 10^9) / (3600 × COP_comp)
    save_power = (cool_energy * 10**9) / (3600 * COP_comp)
    # 步骤3：计算碳减排 C_制冷(Mt) = (E_save × 电网因子) / 10^6
    cooling_col = f"{scheme}_制冷_碳减排量(Mt)"
    df_merge[cooling_col] = (save_power * df_merge["电网碳排放因子"]) / 10**9
    df_merge[cooling_col] = df_merge[cooling_col].round(7)

    # ---------------------- 路径3：余热供电（ORC发电替代电网电） ----------------------
    # 步骤1：计算发电量 E_gen(kWh) = (Q_rec × 10^9 × η_orc) / 3600
    gen_power = (df_merge[reduce_heat_col] * 10**9 * eta_orc) / 3600
    # 步骤2：计算碳减排 C_供电(Mt) = (E_gen × 电网因子) / 10^6
    power_col = f"{scheme}_供电_碳减排量(Mt)"
    df_merge[power_col] = (gen_power * df_merge["电网碳排放因子"]) / 10**9
    df_merge[power_col] = df_merge[power_col].round(7)

# ===================== 5. 整理列顺序 & 保存最终文件 =====================
# 构建最终列顺序：原表列 → 电网因子 → 各方案三路径减排列
base_cols = [
    "年份", "城市名称",
    "方案1_总减少热量(TJ)", "方案2_总减少热量(TJ)", "方案3_总减少热量(TJ)",
    "方案1_总剩余热量(TJ)", "方案2_总剩余热量(TJ)", "方案3_总剩余热量(TJ)",
    "电网碳排放因子"
]

# 追加各方案三路径减排列
add_cols = []
for scheme in scheme_list:
    add_cols.extend([
        f"{scheme}_供暖_碳减排量(Mt)",
        f"{scheme}_制冷_碳减排量(Mt)",
        f"{scheme}_供电_碳减排量(Mt)"
    ])

final_cols = base_cols + add_cols
df_final = df_merge[final_cols].copy()

# 按年份+城市排序
df_final = df_final.sort_values(by=["年份", "城市名称"]).reset_index(drop=True)

# 保存最终Excel文件
df_final.to_excel(output_file, index=False, engine="openpyxl")
print(f"\n✅ 全部计算完成！最终文件已保存至：{output_file}")

# ===================== 6. 结果预览 & 校验 =====================
print("\n===== 2026年 核心结果预览 =====")
preview_cols = [
    "年份", "城市名称", "方案2_总减少热量(TJ)",
    "方案2_供暖_碳减排量(Mt)", "方案2_制冷_碳减排量(Mt)", "方案2_供电_碳减排量(Mt)"
]
print(df_final[df_final["年份"] == 2026][preview_cols].to_string(index=False))

print("\n===== 全量数据统计（方案2 基准情景） =====")
total_heating = df_final["方案2_供暖_碳减排量(Mt)"].sum()
total_cooling = df_final["方案2_制冷_碳减排量(Mt)"].sum()
total_power = df_final["方案2_供电_碳减排量(Mt)"].sum()
print(f"2026-2035年 供暖路径总减排：{total_heating:.4f} Mt")
print(f"2026-2035年 制冷路径总减排：{total_cooling:.4f} Mt")
print(f"2026-2035年 供电路径总减排：{total_power:.4f} Mt")
print(f"同量余热下，供暖减排是制冷的{total_heating/total_cooling:.2f}倍，是供电的{total_heating/total_power:.2f}倍")

===== 开始读取输入文件 =====
✅ 余热表读取完成，共90行数据
包含城市：['Chengdu', 'Chongqing', 'Guiyang', 'Hohhot', 'Qingyang', 'Shaoguan', 'Wuhu', 'Zhangjiakou', 'Zhongwei']
包含年份：[np.int64(2026), np.int64(2027), np.int64(2028), np.int64(2029), np.int64(2030), np.int64(2031), np.int64(2032), np.int64(2033), np.int64(2034), np.int64(2035)]

✅ 电网因子表读取完成，共90行数据
包含城市：['Chengdu', 'Chongqing', 'Guiyang', 'Hohhot', 'Qingyang', 'Shaoguan', 'Wuhu', 'Zhangjiakou', 'Zhongwei']
包含年份：[np.int64(2026), np.int64(2027), np.int64(2028), np.int64(2029), np.int64(2030), np.int64(2031), np.int64(2032), np.int64(2033), np.int64(2034), np.int64(2035)]

✅ 数据匹配完成，合并后共90行数据
电网因子匹配成功率：100.00%

===== 正在计算方案1的三路径碳减排量 =====

===== 正在计算方案2的三路径碳减排量 =====

===== 正在计算方案3的三路径碳减排量 =====

✅ 全部计算完成！最终文件已保存至：D:\杨紫薇\第五篇-算力转移导致的水能碳足迹流动\数据收集与处理\情景二_年份+城市_三路径碳减排量_明细总表.xlsx

===== 2026年 核心结果预览 =====
  年份        城市名称  方案2_总减少热量(TJ)  方案2_供暖_碳减排量(Mt)  方案2_制冷_碳减排量(Mt)  方案2_供电_碳减排量(Mt)
2026     Chengdu     160.813853         0.016784         0.001011         0